In [35]:
# CELL 1: Imports and project configuration

from pathlib import Path
import json
import time
import re
import os
import warnings

import pandas as pd
import pyarrow
import jupyter


warnings.filterwarnings("ignore")

# Project root: current VS Code Jupyter working directory
ROOT = Path.cwd()

print("Current working directory:", ROOT)


# Dataset paths
COUNTERBENCH_DIR = ROOT / "datasets" / "counterbench"
FOLIO_DIR = ROOT / "datasets" / "FOLIO"
RULETAKER_DIR = ROOT / "datasets" / "ruletaker"

# Files visible in your VS Code Explorer
COUNTERBENCH_FILE = (
    COUNTERBENCH_DIR / "data_balanced_alpha_V1.json"
)

FOLIO_TRAIN_FILE = FOLIO_DIR / "folio_v2_train.jsonl"
FOLIO_VALID_FILE = FOLIO_DIR / "folio_v2_validation.jsonl"

RULETAKER_TRAIN_FILE = RULETAKER_DIR / "train.parquet"
RULETAKER_TEST_FILE = RULETAKER_DIR / "test.parquet"
RULETAKER_OOD_FILE = RULETAKER_DIR / "ood_test.parquet"

print("\nPath checks:")

for name, path in [
    ("CounterBench", COUNTERBENCH_FILE),
    ("FOLIO train", FOLIO_TRAIN_FILE),
    ("FOLIO validation", FOLIO_VALID_FILE),
    ("RuleTaker train", RULETAKER_TRAIN_FILE),
    ("RuleTaker test", RULETAKER_TEST_FILE),
    ("RuleTaker OOD test", RULETAKER_OOD_FILE),
]:
    print(f"{name:20s}: {'FOUND' if path.exists() else 'NOT FOUND'}")


Current working directory: c:\Users\SAUMYADIP\OneDrive\Desktop\AI ASSIGNMENT

Path checks:
CounterBench        : FOUND
FOLIO train         : FOUND
FOLIO validation    : FOUND
RuleTaker train     : FOUND
RuleTaker test      : FOUND
RuleTaker OOD test  : FOUND


In [36]:
# CELL 2: Load and inspect the datasets

# 1. Load RuleTaker datasets
ruletaker_train = pd.read_parquet(RULETAKER_TRAIN_FILE)
ruletaker_test = pd.read_parquet(RULETAKER_TEST_FILE)
ruletaker_ood = pd.read_parquet(RULETAKER_OOD_FILE)

# 2. Load FOLIO datasets
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

folio_train = load_jsonl(FOLIO_TRAIN_FILE)
folio_valid = load_jsonl(FOLIO_VALID_FILE)

# 3. Load CounterBench JSON
with open(COUNTERBENCH_FILE, "r", encoding="utf-8") as f:
    counterbench_data = json.load(f)

# 4. Display dataset sizes
print("=" * 60)
print("DATASET SIZES")
print("=" * 60)

print("RuleTaker train:", ruletaker_train.shape)
print("RuleTaker test:", ruletaker_test.shape)
print("RuleTaker OOD:", ruletaker_ood.shape)

print("FOLIO train:", len(folio_train))
print("FOLIO validation:", len(folio_valid))

if isinstance(counterbench_data, list):
    print("CounterBench examples:", len(counterbench_data))
elif isinstance(counterbench_data, dict):
    print("CounterBench top-level keys:", list(counterbench_data.keys()))
else:
    print("CounterBench data type:", type(counterbench_data))

# 5. Inspect actual schemas
print("\n" + "=" * 60)
print("RULETAKER COLUMNS")
print("=" * 60)
print(ruletaker_train.columns.tolist())

print("\nRULETAKER SAMPLE")
display(ruletaker_train.head(3))

print("\nFOLIO SAMPLE")
for example in folio_train[:2]:
    print(json.dumps(example, indent=2, ensure_ascii=False))

print("\nCOUNTERBENCH SAMPLE")
if isinstance(counterbench_data, list):
    print(json.dumps(counterbench_data[:2], indent=2, ensure_ascii=False))
elif isinstance(counterbench_data, dict):
    for key, value in list(counterbench_data.items())[:3]:
        print(f"\nKey: {key}")
        print(str(value)[:1500])


DATASET SIZES
RuleTaker train: (1000, 5)
RuleTaker test: (250, 5)
RuleTaker OOD: (375, 5)
FOLIO train: 1001
FOLIO validation: 203
CounterBench examples: 1000

RULETAKER COLUMNS
['context', 'statement', 'reasoning', 'depth', 'flag']

RULETAKER SAMPLE


,context,statement,reasoning,depth,flag
0,Facts:\nThe wall dances gracefully.\nThe river...,The sandwich dances gracefully.,Since we know that based on the river dances g...,2,False
1,Facts:\nCharlie is fast.\nStanford University ...,The river volunteers.,"Given that given that charlie is fast, it foll...",3,False
2,Facts:\nThe globe is colorful.\nThe globe is f...,Mike is colorful.,Applied Rule: since the fact that if the globe...,1,True



FOLIO SAMPLE
{
  "story_id": 406,
  "premises": "All people who regularly drink coffee are dependent on caffeine.\nPeople regularly drink coffee, or they don't want to be addicted to caffeine, or both.\nNo one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.\nRina is either a student who is unaware that caffeine is a drug, or she is not a student and is she aware that caffeine is a drug.\nRina  is either a student who is dependent on caffeine, or she is not a student and not dependent on caffeine.",
  "premises-FOL": "∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))\n∀x (DrinkRegularly(x, coffee)  ∨ (¬WantToBeAddictedTo(x, caffeine)))\n∀x (¬WantToBeAddictedTo(x, caffeine) → ¬AwareThatDrug(x, caffeine))\n¬(Student(rina) ⊕  ¬AwareThatDrug(rina, caffeine))\n¬(IsDependentOn(rina, caffeine) ⊕ Student(rina))",
  "conclusion": "Rina doesn't want to be addicted to caffeine or is unaware that caffeine is a drug.",
  "conclusion-FOL": "¬WantToBeAddicted

In [37]:
# CELL 3: backward-chaining proof search (BFS / DFS / UCS / A*) over Horn rules
import heapq, itertools, re

def horn_from_ruletaker(context):
    """Parse a (normalised) RuleTaker context into ground facts and single-premise Horn rules over string atoms."""
    T = re.compile(r"^(?:given that|based on|assuming|when|if)\s+(.*?),\s*(?:it follows that|it is evident that|we can say|we can conclude|then)\s+(.*?)\.?\s*$", re.I)
    norm = lambda x: re.sub(r"\s+", " ", re.sub(r"^the\s+", "", x.lower().strip().rstrip(".")))
    facts, rules = set(), []
    for l in str(context).split("\n"):
        l = l.strip()
        if not l or l.lower() in ("facts:", "rules:"): continue
        m = T.match(l)
        if m: rules.append((norm(m.group(1)), norm(m.group(2))))
        else: facts.add(norm(l))
    return facts, rules

def prove(goal, facts, rules, strategy="astar", max_nodes=20000):
    """Backward-chaining proof search as a state-space search.
    state = frozenset of open sub-goals; a move replaces one goal by the body of a rule whose head it is
    (or discards it if it is a known fact). Goal state = no open sub-goals.
      bfs   : FIFO frontier            dfs : LIFO frontier
      ucs   : priority = g (rules used)    astar : priority = g + h,  h = number of open sub-goals."""
    by_head = {}
    for b, h in rules: by_head.setdefault(h, []).append(b)
    start = frozenset([goal]); tick = itertools.count(); seen = {start}; nodes = 0
    frontier = [(0, next(tick), 0, start)] if strategy in ("ucs", "astar") else None
    queue = [(0, start)]
    while (frontier if frontier is not None else queue):
        if frontier is not None: _, _, g, st = heapq.heappop(frontier)
        elif strategy == "bfs": g, st = queue.pop(0)
        else: g, st = queue.pop()
        nodes += 1
        if nodes > max_nodes: return False, nodes
        if not st: return True, nodes
        goal_ = sorted(st)[0]; rest = st - {goal_}
        nxt = []
        if goal_ in facts: nxt.append(rest)
        for body in by_head.get(goal_, []): nxt.append(rest | {body})
        for n in nxt:
            if n in seen: continue
            seen.add(n)
            if frontier is not None:
                h = len(n) if strategy == "astar" else 0
                heapq.heappush(frontier, (g + 1 + h, next(tick), g + 1, n))
            else: queue.append((g + 1, n))
    return False, nodes


In [38]:
# CELL 4: first-order logic - parser, CNF, unification, resolution (BFS/DFS/A*), DPLL
import re, time, itertools, heapq
from collections import defaultdict

# =====================================================================
# 1. FOL parser  (∀ ∃ ¬ ∧ ∨ ⊕ → ↔ = ≠, predicates, functions)
# =====================================================================
_TOK = re.compile(r"∀|∃|¬|∧|∨|⊕|→|↔|⟷|≠|=|\(|\)|,|[A-Za-z0-9_][A-Za-z0-9_'\-\.]*")
_QUANT = {"∀": "all", "∃": "ex"}

class FOLSyntaxError(Exception): pass
EXACTLY_ONE = True

def _tokenize(s):
    s = s.replace("\u2014", "-").strip()
    return _TOK.findall(s)

class _Parser:
    def __init__(self, text):
        self.t = _tokenize(text); self.i = 0; self.bound = []; self.free = set(); self.fuzzy_free = True
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        tok = self.peek()
        if tok is None or (x is not None and tok != x): raise FOLSyntaxError(f"expected {x} got {tok}")
        self.i += 1; return tok
    def parse(self):
        f = self.iff()
        if self.peek() is not None: raise FOLSyntaxError("trailing tokens")
        for v in sorted(self.free): f = ("all", v, f)          # Prover9 convention: free variables are universal
        return f
    def iff(self):
        a = self.imp()
        while self.peek() in ("↔", "⟷"):
            self.eat(); a = ("iff", a, self.imp())
        return a
    def imp(self):
        a = self.orx()
        if self.peek() == "→":
            self.eat(); return ("imp", a, self.imp())
        return a
    def orx(self):
        a = self.andx(); xs = [a]; only_xor = True
        while self.peek() in ("∨", "⊕"):
            op = self.eat(); b = self.andx()
            if op == "∨": a = ("or", a, b); only_xor = False; xs = [a]
            else:
                xs.append(b); a = ("xor", a, b)
        if only_xor and len(xs) > 2 and EXACTLY_ONE:            # a ⊕ b ⊕ c ... read as "exactly one of"
            atleast = xs[0]
            for y in xs[1:]: atleast = ("or", atleast, y)
            f = atleast
            for i in range(len(xs)):
                for j in range(i + 1, len(xs)):
                    f = ("and", f, ("or", ("not", xs[i]), ("not", xs[j])))
            return f
        return a
    def andx(self):
        a = self.unary()
        while self.peek() == "∧":
            self.eat(); a = ("and", a, self.unary())
        return a
    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.eat(); return ("not", self.unary())
        if tok in _QUANT:
            self.eat(); vs = []
            while self.peek() is not None and re.match(r"[A-Za-z_]", self.peek()) and self.peek() not in _QUANT:
                nxt = self.t[self.i + 1] if self.i + 1 < len(self.t) else None
                if vs and nxt == "(":   # a later identifier followed by "(" starts the body (a predicate)
                    break
                vs.append(self.eat())
                if self.peek() == ",": self.eat()
            if not vs: raise FOLSyntaxError("quantifier without variable")
            self.bound.extend(v.lower() for v in vs)
            body = self.unary()
            for _ in vs: self.bound.pop()
            for v in reversed(vs): body = (_QUANT[tok], v.lower(), body)
            return body
        if tok == "(":
            self.eat(); f = self.iff(); self.eat(")")
            # parenthesised term followed by = ? (rare) -> not supported
            return f
        return self.atom()
    def term(self):
        name = self.eat()
        if not re.match(r"[A-Za-z0-9_]", name): raise FOLSyntaxError("bad term " + name)
        low = name.lower()
        if self.peek() == "(":
            self.eat(); args = [self.term()]
            while self.peek() == ",": self.eat(); args.append(self.term())
            self.eat(")"); return (low,) + tuple(args)
        if low in self.bound: return "?" + low
        if self.fuzzy_free and re.fullmatch(r"[u-z]\d*", low): self.free.add(low); return "?" + low
        return low
    def atom(self):
        t = self.term()
        if self.peek() in ("=", "≠"):
            op = self.eat(); u = self.term(); a = ("atom", "eq", (t, u))
            return a if op == "=" else ("not", a)
        if isinstance(t, tuple): return ("atom", t[0], t[1:])
        if t.startswith("?"): raise FOLSyntaxError("variable used as proposition")
        return ("atom", t, ())

def parse_fol(text):
    return _Parser(text).parse()

def _balance(tokens):
    d = 0
    for t in tokens:
        d += (t == "(") - (t == ")")
    return d

def _try_parse_tokens(tokens):
    p = _Parser(""); p.t = tokens
    try: return p.parse()
    except (FOLSyntaxError, RecursionError, IndexError): return None

def _score(f):
    """prefer repairs that give the usual shape  forall x (A -> B)  /  A -> B  at the top."""
    while f[0] in ("all", "ex"): f = f[2]
    return 0 if f[0] in ("imp", "iff") else 1

def parse_fol_tolerant(text):
    """Parse; if parentheses are unbalanced, try the most plausible repair (the gold FOL contains such typos)."""
    toks = _tokenize(text)
    f = _try_parse_tokens(toks)
    if f is not None: return f
    d = _balance(toks)
    cands = []
    if d > 0:      # missing ")" : insert at every position from the end backwards
        for pos in range(len(toks), 0, -1):
            cands.append(toks[:pos] + [")"] + toks[pos:])
        extra = d - 1
        if extra > 0:
            cands = [c + [")"] * extra for c in cands]
    elif d < 0:    # surplus ")" : drop one from the right
        idx = [i for i, t in enumerate(toks) if t == ")"]
        for i in reversed(idx):
            c = toks[:i] + toks[i + 1:]
            for _ in range(-d - 1):
                j = max((k for k, t in enumerate(c) if t == ")"), default=None)
                if j is not None: c = c[:j] + c[j + 1:]
            cands.append(c)
    best = None
    for c in cands:
        g = _try_parse_tokens(c)
        if g is None: continue
        sc = _score(g)
        if best is None or sc < best[0]: best = (sc, g)
        if sc == 0: break
    if best: return best[1]
    # last resort: trailing junk
    for cut in range(len(toks) - 1, 2, -1):
        g = _try_parse_tokens(toks[:cut] + [")"] * max(0, _balance(toks[:cut])))
        if g is not None: return g
    raise FOLSyntaxError("unparsable")

# =====================================================================
# 2. Clause normal form (eliminate -> <-> xor, NNF, Skolemise, distribute)
# =====================================================================
_sk = itertools.count(1); _vr = itertools.count(1)

def _elim(f):
    k = f[0]
    if k == "atom": return f
    if k == "not": return ("not", _elim(f[1]))
    if k in ("and", "or"): return (k, _elim(f[1]), _elim(f[2]))
    if k == "imp": return ("or", ("not", _elim(f[1])), _elim(f[2]))
    if k == "iff":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    if k == "xor":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (k, f[1], _elim(f[2]))

def _nnf(f, neg=False):
    k = f[0]
    if k == "atom": return ("not", f) if neg else f
    if k == "not": return _nnf(f[1], not neg)
    if k in ("and", "or"):
        kk = {"and": "or", "or": "and"}[k] if neg else k
        return (kk, _nnf(f[1], neg), _nnf(f[2], neg))
    kk = {"all": "ex", "ex": "all"}[k] if neg else k
    return (kk, f[1], _nnf(f[2], neg))

def _subst_term(t, m):
    if isinstance(t, tuple): return (t[0],) + tuple(_subst_term(a, m) for a in t[1:])
    return m.get(t, t)

def _skolem(f, env, univ):
    k = f[0]
    if k == "atom": return ("atom", f[1], tuple(_subst_term(a, env) for a in f[2]))
    if k == "not": return ("not", _skolem(f[1], env, univ))
    if k in ("and", "or"): return (k, _skolem(f[1], env, univ), _skolem(f[2], env, univ))
    v = "?" + f[1]
    if k == "all":
        nv = f"?v{next(_vr)}"; e = dict(env); e[v] = nv
        return _skolem(f[2], e, univ + [nv])
    e = dict(env)
    e[v] = (f"sk{next(_sk)}",) + tuple(univ) if univ else f"sk{next(_sk)}"
    return _skolem(f[2], e, univ)

def _cnf(f):
    k = f[0]
    if k == "and": return _cnf(f[1]) + _cnf(f[2])
    if k == "or":
        a, b = _cnf(f[1]), _cnf(f[2])
        if len(a) * len(b) > 4000: raise FOLSyntaxError("CNF blow-up")
        return [x | y for x in a for y in b]
    if k == "not": return [frozenset([(False, f[1][1], f[1][2])])]
    return [frozenset([(True, f[1], f[2])])]

def to_clauses(f):
    g = _skolem(_nnf(_elim(f)), {}, [])
    out = []
    for c in _cnf(g):
        if any((not s, p, a) in c for (s, p, a) in c): continue        # tautology
        out.append(c)
    return out

# =====================================================================
# 3. Unification + resolution (given-clause loop, pluggable search strategy)
# =====================================================================
def _isvar(t): return isinstance(t, str) and t.startswith("?")

def _walk(t, s):
    while _isvar(t) and t in s: t = s[t]
    return t

def _occurs(v, t, s):
    t = _walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(_occurs(v, a, s) for a in t[1:])

def unify(a, b, s):
    a, b = _walk(a, s), _walk(b, s)
    if a == b: return s
    if _isvar(a):
        if _occurs(a, b, s): return None
        s = dict(s); s[a] = b; return s
    if _isvar(b): return unify(b, a, s)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def _apply(t, s):
    t = _walk(t, s)
    if isinstance(t, tuple): return (t[0],) + tuple(_apply(a, s) for a in t[1:])
    return t

def _size(t): return 1 + sum(_size(a) for a in t[1:]) if isinstance(t, tuple) else 1

def _canon(clause):
    """canonical key of a clause up to variable renaming (for duplicate detection)."""
    m = {}
    def ren(t):
        if isinstance(t, tuple): return (t[0],) + tuple(ren(a) for a in t[1:])
        if _isvar(t):
            if t not in m: m[t] = f"?{len(m)}"
            return m[t]
        return t
    return tuple(sorted(((s, p, tuple(ren(a) for a in args)) for s, p, args in sorted(clause, key=repr)), key=repr))

_rn = itertools.count(1)
def _rename(clause):
    m = {}; suf = next(_rn)
    def ren(t):
        if isinstance(t, tuple): return (t[0],) + tuple(ren(a) for a in t[1:])
        if _isvar(t):
            if t not in m: m[t] = f"{t.split('#')[0]}#{suf}"
            return m[t]
        return t
    return tuple((s, p, tuple(ren(a) for a in args)) for s, p, args in clause)

def _factor(lits):
    """return the clause plus all single-step factors (merge two unifiable same-sign literals)."""
    outs = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = {}
                for x, y in zip(a[2], b[2]):
                    s = unify(x, y, s)
                    if s is None: break
                if s is not None:
                    outs.append(tuple(dict.fromkeys((sg, p, tuple(_apply(t, s) for t in args)) for sg, p, args in lits)))
    return outs

class Prover:
    """Resolution refutation. strategy: 'bfs' | 'dfs' | 'astar' decides which clause is expanded next."""
    def __init__(self, strategy="astar", max_given=3000, time_limit=1.0, max_lits=8, max_term=12):
        self.strategy, self.max_given, self.time_limit, self.max_lits, self.max_term = strategy, max_given, time_limit, max_lits, max_term
        self.stats = {"given": 0, "generated": 0}

    def _key(self, clause, depth, tick):
        if self.strategy == "bfs": return (depth, tick)                    # Breadth-first: shallowest first
        if self.strategy == "dfs": return (-depth, -tick)                   # Depth-first: deepest/newest first
        h = sum(1 + sum(_size(a) for a in args) for _, _, args in clause)   # A*: f = g + h
        return (depth + h, tick)

    def refute(self, kb_clauses, goal_clauses, trace=None):
        t0 = time.perf_counter(); tick = itertools.count()
        usable = defaultdict(list)       # (sign,pred) -> list of clause tuples that can be resolved against
        seen = set(); heap = []
        info = {}                         # clause -> (parents, rule)
        def add_usable(c):
            for l in c: usable[(l[0], l[1])].append(c)
        for c in kb_clauses:
            c = tuple(c); k = _canon(c)
            if k in seen: continue
            seen.add(k); add_usable(c); info[c] = ("premise", ())
        for c in goal_clauses:
            c = tuple(c); k = _canon(c)
            if k in seen: continue
            seen.add(k); info[c] = ("negated goal", ())
            heapq.heappush(heap, (self._key(c, 0, next(tick)), next(tick), 0, c))
        while heap and self.stats["given"] < self.max_given:
            if time.perf_counter() - t0 > self.time_limit: return None
            _, _, depth, given = heapq.heappop(heap)
            self.stats["given"] += 1
            add_usable(given)
            g = _rename(given)
            for gi, (gs, gp, ga) in enumerate(g):
                for other in list(usable[(not gs, gp)]):
                    o = _rename(other)
                    for oi, (os_, op, oa) in enumerate(o):
                        if os_ == gs or op != gp or len(oa) != len(ga): continue
                        s = {}
                        for x, y in zip(ga, oa):
                            s = unify(x, y, s)
                            if s is None: break
                        if s is None: continue
                        rest = [l for j, l in enumerate(g) if j != gi] + [l for j, l in enumerate(o) if j != oi]
                        res = tuple(dict.fromkeys((sg, p, tuple(_apply(t, s) for t in args)) for sg, p, args in rest))
                        self.stats["generated"] += 1
                        if not res:
                            if trace is not None: trace.append((given, other, "contradiction (empty clause)"))
                            return True
                        if len(res) > self.max_lits: continue
                        if any(_size(t) > self.max_term for _, _, a in res for t in a): continue
                        if any((not sg, p, a) in res for sg, p, a in res): continue          # tautology
                        for cand in [res] + _factor(res):
                            k = _canon(cand)
                            if k in seen: continue
                            seen.add(k); info[cand] = ("resolve", (given, other))
                            heapq.heappush(heap, (self._key(cand, depth + 1, next(tick)), next(tick), depth + 1, cand))
        return None if heap else False      # None = gave up (limit), False = saturated without proof

def _fmt_term(t):
    if isinstance(t, tuple): return f"{t[0]}({', '.join(_fmt_term(a) for a in t[1:])})"
    return t
def fmt_clause(c): return " v ".join(("" if s else "~") + p + (f"({', '.join(_fmt_term(a) for a in args)})" if args else "") for s, p, args in c) or "[]"

# =====================================================================
# 4. Ground DPLL (propositional SAT) on the Herbrand instances
# =====================================================================
def dpll(clauses, assignment=None):
    """clauses: list of sets of ints (DIMACS-style). Returns a model dict or None (unsatisfiable)."""
    assignment = dict(assignment or {})
    clauses = [set(c) for c in clauses]
    while True:                                                           # unit propagation
        unit = next((next(iter(c)) for c in clauses if len(c) == 1), None)
        if unit is None: break
        assignment[abs(unit)] = unit > 0
        new = []
        for c in clauses:
            if unit in c: continue
            if -unit in c:
                c = c - {-unit}
                if not c: return None
            new.append(c)
        clauses = new
    if not clauses: return assignment
    lits = {l for c in clauses for l in c}
    for l in list(lits):                                                  # pure literal elimination
        if -l not in lits:
            assignment[abs(l)] = l > 0
            clauses = [c for c in clauses if l not in c]
    if not clauses: return assignment
    cnt = defaultdict(int)
    for c in clauses:
        for l in c: cnt[abs(l)] += 1
    v = max(cnt, key=cnt.get)                                             # branching heuristic (most frequent var)
    for val in (v, -v):
        r = dpll(clauses + [{val}], assignment)
        if r is not None: return r
    return None

def _ground_terms(consts, funcs, depth):
    terms = set(consts)
    for _ in range(depth):
        terms |= {(f,) + args for f, n in funcs for args in itertools.product(sorted(terms, key=repr), repeat=n)}
    return sorted(terms, key=repr)

def _vars_of(t, out):
    if _isvar(t): out.add(t)
    elif isinstance(t, tuple):
        for a in t[1:]: _vars_of(a, out)

def dpll_refute(kb_clauses, goal_clauses, depth=1, max_ground=6000):
    """Ground the clause set over the Herbrand universe (depth-limited) and run DPLL.
    UNSAT  => KB entails the goal (sound).   SAT/limit => unknown (False)."""
    allc = [tuple(c) for c in kb_clauses] + [tuple(c) for c in goal_clauses]
    consts, funcs = set(), set()
    def scan(t):
        if isinstance(t, tuple):
            funcs.add((t[0], len(t) - 1)); [scan(a) for a in t[1:]]
        elif not _isvar(t): consts.add(t)
    for c in allc:
        for _, _, a in c: [scan(t) for t in a]
    if not consts: consts.add("c0")
    U = _ground_terms(consts, funcs, depth)
    ids = {}
    def lid(s, p, args):
        k = (p, args)
        if k not in ids: ids[k] = len(ids) + 1
        return ids[k] if s else -ids[k]
    out = []
    for c in allc:
        vs = set()
        for _, _, a in c: [_vars_of(t, vs) for t in a]
        vs = sorted(vs)
        if len(U) ** len(vs) > max_ground: return None
        for combo in itertools.product(U, repeat=len(vs)):
            m = dict(zip(vs, combo))
            out.append({lid(s, p, tuple(_subst_term(t, m) for t in a)) for s, p, a in c})
            if len(out) > max_ground: return None
    return dpll(out) is None


import difflib
FUZZY = True
def _collect(f, preds, consts):
    k = f[0]
    if k == "atom":
        preds.add((f[1], len(f[2])))
        def walk(t):
            if isinstance(t, tuple): preds_f.add((t[0], len(t) - 1)); [walk(a) for a in t[1:]]
            elif not t.startswith("?"): consts.add(t)
        for a in f[2]: walk(a)
    elif k == "not": _collect(f[1], preds, consts)
    elif k in ("and", "or", "imp", "iff", "xor"): _collect(f[1], preds, consts); _collect(f[2], preds, consts)
    else: _collect(f[2], preds, consts)
preds_f = set()

def _merge_map(names, ratio=0.85):
    names = sorted(names, key=lambda n: (-len(n), n)); m = {}
    for i, a in enumerate(names):
        if a in m: continue
        for b in names[i + 1:]:
            if b in m or len(a) < 4 or len(b) < 4: continue
            if difflib.SequenceMatcher(None, a, b).ratio() >= ratio: m[b] = a
    return m

def _rename_ast(f, pm, cm):
    k = f[0]
    def rt(t):
        if isinstance(t, tuple): return (t[0],) + tuple(rt(a) for a in t[1:])
        return t if t.startswith("?") else cm.get(t, t)
    if k == "atom": return ("atom", pm.get(f[1], f[1]), tuple(rt(a) for a in f[2]))
    if k == "not": return ("not", _rename_ast(f[1], pm, cm))
    if k in ("and", "or", "imp", "iff", "xor"): return (k, _rename_ast(f[1], pm, cm), _rename_ast(f[2], pm, cm))
    return (k, f[1], _rename_ast(f[2], pm, cm))

def _unify_symbols(asts):
    preds, consts = set(), set()
    for a in asts: _collect(a, preds, consts)
    pm = _merge_map({p for p, _ in preds} - {"eq"})
    cm = _merge_map(consts)
    if not pm and not cm: return asts
    return [_rename_ast(a, pm, cm) for a in asts]

# =====================================================================
# 5. Public API
# =====================================================================
def _split_formulas(text):
    return [l.strip() for l in re.split(r"\n+", text) if l.strip()]

def looks_like_fol(text):
    return bool(re.search(r"[∀∃∧∨→¬⊕]", text))

def fol_entails(premises, conclusion, method="auto", strategy="astar", time_limit=1.0, trace=None):
    """premises: str (one formula per line) or list; conclusion: str. True iff premises |= conclusion."""
    plist = _split_formulas(premises) if isinstance(premises, str) else list(premises)
    asts = []
    for p in plist:
        try: asts.append(parse_fol_tolerant(p))
        except (FOLSyntaxError, RecursionError, IndexError): continue      # skip malformed premise
    try: g_ast = parse_fol_tolerant(conclusion)
    except (FOLSyntaxError, RecursionError, IndexError): return False
    allast = asts + [g_ast]
    if FUZZY: allast = _unify_symbols(allast)
    asts, g_ast = allast[:-1], allast[-1]
    kb = []
    for a in asts:
        try: kb += to_clauses(a)
        except (FOLSyntaxError, RecursionError): continue
    try: goal = to_clauses(("not", g_ast))
    except (FOLSyntaxError, RecursionError): return False
    kb.append(frozenset([(True, "eq", ("?x", "?x"))]))
    kb.append(frozenset([(False, "eq", ("?x", "?y")), (True, "eq", ("?y", "?x"))]))
    if method in ("dpll", "auto"):                      # fast ground SAT first
        r = dpll_refute(kb, goal, depth=1)
        if r is not None or method == "dpll":
            return bool(r)                              # (auto: None = grounding too large -> fall through to resolution)
    P = Prover(strategy=strategy, time_limit=time_limit)
    r = P.refute(kb, goal, trace)
    fol_entails.last_stats = P.stats
    return r is True


In [39]:
# CELL 5: counterfactual reasoning over structural causal models (CounterBench)
import re, itertools

def scm_parse(given, question):
    """Parse a CounterBench prompt into (mechanisms, priors, evidence, factual, interventions, target)."""
    g = given.strip()
    m = re.search(r"We know that\s*(.*?)\.\s*(?=[A-Z][a-z]*\s*~|\s*We observed|$)", g + " ", re.S)
    body = m.group(1) if m else g
    # priors:  Blaf~ Bern(0.2)  /  Blaf ~Bern(0.7)
    priors = {v: float(p) for v, p in re.findall(r"([A-Z][a-z]+)\s*~\s*Bern\(([\d.]+)\)", g)}
    # evidence: after "We observed"
    ev = {}
    mo = re.search(r"We observed\s*(.*)$", g, re.S)
    if mo:
        txt = mo.group(1).strip().rstrip(".")
        for part in re.split(r",|\band\b", txt):
            names = re.findall(r"\b[A-Z][a-z]+\b", part)
            if names:
                neg = bool(re.search(r"\bnot\b", part, re.I)) or bool(re.search(r"\bdoes not\b|\bnot occur", part, re.I))
                ev[names[0]] = 0 if neg else 1
    # ---- mechanisms ----
    segs = [s.strip() for s in re.split(r",", body) if s.strip()]
    clauses = []   # [left_expr, effects]; left_expr = (op, [(var,neg)...])
    for s in segs:
        s = re.sub(r"^and\s+", "", s).strip()
        mm = re.match(r"^(.*?)\s+(?:together\s+)?(causes?)\s+(.*)$", s)
        if mm:
            lhs, rhs = mm.group(1), mm.group(3)
            op = "or" if re.search(r"\bor\b", lhs) else "and"
            lits = [(re.findall(r"[A-Z][a-z]+", p)[0], bool(re.search(r"\bnot\b", p, re.I)))
                    for p in re.split(r"\s+(?:and|or)\s+", lhs.replace("together", "").strip())]
            effs = [(re.findall(r"[A-Z][a-z]+", p)[0], bool(re.search(r"\bnot\b", p, re.I)))
                    for p in re.split(r"\s+and\s+", rhs.strip())]
            clauses.append([(op, lits), effs])
        elif clauses:   # continuation of the effect list: "Trune and Vork"
            for p in re.split(r"\s+and\s+", s):
                nm = re.findall(r"[A-Z][a-z]+", p)
                if nm: clauses[-1][1].append((nm[0], bool(re.search(r"\bnot\b", p, re.I))))
    mech = {}   # effect -> list of (op, lits, neg_effect)
    for (op, lits), effs in clauses:
        for v, neg in effs:
            mech.setdefault(v, []).append((op, lits, neg))
    # ---- question ----
    q = question.strip()
    target = re.findall(r"Would\s+([A-Z][a-z]+)\s+occur", q)
    target = target[0] if target else None
    do, factual = {}, {}
    body_q = re.sub(r"^.*?(?=Would)", "", q) if q.startswith("Assume") else q
    if q.startswith("Assume"):
        for part in re.findall(r"(not\s+)?([A-Z][a-z]+)", q.split("Would")[0]):
            if part[1] in ("Assume",): continue
            do[part[1]] = 0 if part[0] else 1
    else:
        mi = re.search(r"if\s+(.*?)\s+instead of\s+([A-Z][a-z]+)", q)
        if mi:
            nm = re.findall(r"(not\s+)?([A-Z][a-z]+)", mi.group(1))
            for neg, v in nm: do[v] = 0 if neg else 1
            factual[mi.group(2)] = 1 - (0 if nm[0][0] else 1) if False else 1
        else:
            mj = re.search(r"if\s+(.*?)\?", q)
            if mj:
                for neg, v in re.findall(r"(not\s+)?([A-Z][a-z]+)", mj.group(1)):
                    do[v] = 0 if neg else 1
    return mech, priors, ev, factual, do, target

def scm_answer(given, question):
    mech, priors, ev, factual, do, target = scm_parse(given, question)
    names = set(mech) | {l[0] for ms in mech.values() for (_, ls, _) in ms for l in ls} | set(ev) | set(do) | set(factual) | {target}
    roots = sorted(v for v in names if v not in mech)

    allvars = sorted(names - set(roots))
    def evaluate(u, interv):
        # fixed-point evaluation (handles cyclic mechanisms): iterate synchronous updates from 0
        cur = {v: 0 for v in allvars}; cur.update(u); cur.update(interv)
        for _ in range(len(allvars) + 3):
            changed = False
            for v in allvars:
                if v in interv: continue
                out = 0
                for op, lits, neg in mech[v]:
                    xs = [cur.get(a, 0) ^ int(n) for a, n in lits]
                    r = all(xs) if op == "and" else any(xs)
                    out |= (int(r) ^ int(neg))
                if cur[v] != out: cur[v] = out; changed = True
            if not changed: break
        return lambda v: cur.get(v, 0)

    def run(use_factual):
        num = den = 0.0
        for bits in itertools.product([0, 1], repeat=len(roots)):          # DFS/enumeration of exogenous worlds
            u = dict(zip(roots, bits))
            w = 1.0
            for r_, b in u.items():
                p = priors.get(r_, 0.5); w *= p if b else 1 - p
            fv = evaluate(u, {})
            if any(fv(v) != x for v, x in ev.items()): continue
            if use_factual and any(fv(v) != x for v, x in factual.items()): continue
            cv = evaluate(u, do)
            den += w; num += w * cv(target)
        return num, den
    num, den = run(True)
    if den == 0: num, den = run(False)
    return (num / den > 0.5) if den else False


In [40]:
# CELL 6: Unified predictor + evaluation on all public data

import os

_RULE_TEMPLATE = re.compile(
    r"^(?:given that|based on|assuming|when|if)\s+(.*?),\s*"
    r"(?:it follows that|it is evident that|we can say|we can conclude|then)\s+(.*?)\.?\s*$",
    re.IGNORECASE,
)

def normalise_context(context):
    """Drop 'Facts:'/'Rules:' headers and rewrite the 5 paraphrase templates to 'If X, then Y.'"""
    out = []
    for line in str(context).split("\n"):
        line = line.strip()
        if not line or line.lower() in ("facts:", "rules:"):
            continue
        m = _RULE_TEMPLATE.match(line)
        out.append(f"If {m.group(1)}, then {m.group(2)}." if m else (line if line.endswith((".", "?", "!")) else line + "."))
    return " ".join(out)

def predict(context=None, statement=None, trace=False, premises_fol=None, conclusion_fol=None, fol_method="auto"):
    """Entailment check -> bool.  Routing:
         1. logic given directly (or FOL symbols in the text)  -> FOL prover (DPLL / resolution)
         2. counterfactual prompt with Bernoulli priors        -> structural-causal-model solver
         3. everything else (rules in natural language)        -> parser + forward chaining engine (CELL 3)"""
    try:
        if premises_fol is not None and conclusion_fol is not None:
            return bool(fol_entails(premises_fol, conclusion_fol, method=fol_method))
        if context is not None and statement is not None and looks_like_fol(str(context)) and looks_like_fol(str(statement)):
            return bool(fol_entails(str(context), str(statement), method=fol_method))
        if "Bern(" in str(context) or re.search(r"Bern\s*\(", str(context)):
            return bool(scm_answer(str(context), str(statement)))
        res = v_infer_answer(normalise_context(context), statement, trace=trace)
        return (bool(res[0]), res[1]) if trace else bool(res[0] if isinstance(res, tuple) else res)
    except Exception as err:                       # never crash a batch run
        return (False, [f"(error: {err})"]) if trace else False

def evaluate(name, rows):
    """rows: list of (kwargs-for-predict, gold_bool). Prints accuracy and total inference time."""
    t0 = time.perf_counter()
    preds = [predict(**kw) for kw, _ in rows]
    dt = time.perf_counter() - t0
    correct = sum(p == g for p, (_, g) in zip(preds, rows))
    print(f"{name:38s} {correct:5d}/{len(rows):<5d} = {100*correct/len(rows):6.2f}%   time = {dt:6.2f}s")
    return correct, len(rows), dt

# ---- evaluation sets built from the public data loaded in CELL 2 ----
eval_sets = {}
for split, df in [("RuleTaker train", ruletaker_train), ("RuleTaker test", ruletaker_test), ("RuleTaker OOD test", ruletaker_ood)]:
    eval_sets[split] = [(dict(context=c, statement=s), f == "True") for c, s, f in zip(df.context, df.statement, df.flag)]

# FOLIO: the logic is given directly (premises-FOL / conclusion-FOL). Binary task: entailed = True, otherwise False.
def folio_rows(data):
    return [(dict(premises_fol=x["premises-FOL"], conclusion_fol=x["conclusion-FOL"]), x["label"] == "True") for x in data]
eval_sets["FOLIO validation (FOL given)"] = folio_rows(folio_valid)
eval_sets["FOLIO train (FOL given)"] = folio_rows(folio_train)

def cb_rows(data):
    return [(dict(context=x.get("given_info", ""), statement=x["question"]), str(x["answer"]).strip().lower() in ("yes", "true", "1")) for x in data]
eval_sets["CounterBench alpha"] = cb_rows(counterbench_data)
_backdoor = Path(COUNTERBENCH_FILE).parent / "data_balanced_backdoor_V2.json"
if _backdoor.exists():
    with open(_backdoor, "r", encoding="utf-8") as f:
        eval_sets["CounterBench backdoor"] = cb_rows(json.load(f))

print("=" * 90)
print("PUBLIC-DATASET RESULTS")
print("=" * 90)
results = {name: evaluate(name, rows) for name, rows in eval_sets.items()}
tot_c = sum(r[0] for r in results.values()); tot_n = sum(r[1] for r in results.values()); tot_t = sum(r[2] for r in results.values())
print("-" * 90)
print(f"{'OVERALL':38s} {tot_c:5d}/{tot_n:<5d} = {100*tot_c/tot_n:6.2f}%   time = {tot_t:6.2f}s")


PUBLIC-DATASET RESULTS
RuleTaker train                          983/1000  =  98.30%   time =   0.28s
RuleTaker test                           244/250   =  97.60%   time =   0.07s
RuleTaker OOD test                       373/375   =  99.47%   time =   0.09s
FOLIO validation (FOL given)             178/203   =  87.68%   time =   6.70s
FOLIO train (FOL given)                  809/1001  =  80.82%   time =  19.95s
CounterBench alpha                       960/1000  =  96.00%   time =   0.14s
CounterBench backdoor                    180/200   =  90.00%   time =   0.02s
------------------------------------------------------------------------------------------
OVERALL                                 3727/4029  =  92.50%   time =  27.24s


## 5. Comparison of the search strategies

In [41]:
# CELL 7: BFS vs DFS vs UCS vs A*  (backward-chaining proof search), and resolution strategies vs DPLL

print("Backward-chaining state-space search on RuleTaker test (accuracy / avg nodes expanded / time)")
for strat in ["bfs", "dfs", "ucs", "astar"]:
    t0 = time.perf_counter(); ok = nodes = 0
    for c, q, f in zip(ruletaker_test.context, ruletaker_test.statement, ruletaker_test.flag):
        facts, rules = horn_from_ruletaker(c); goal = list(horn_from_ruletaker(q)[0])[0]
        r, n = prove(goal, facts, rules, strat); ok += (r == (f == "True")); nodes += n
    print(f"  {strat:6s} {100*ok/len(ruletaker_test):6.2f}%   {nodes/len(ruletaker_test):5.1f} nodes   {time.perf_counter()-t0:5.2f}s")

print("\nFOLIO validation (logic given): resolution with different clause-selection strategies vs DPLL")
for name, kw in [("Resolution + A*", dict(method="resolution", strategy="astar")), ("Resolution + BFS", dict(method="resolution", strategy="bfs")),
                 ("Resolution + DFS", dict(method="resolution", strategy="dfs")), ("DPLL (ground SAT)", dict(method="dpll"))]:
    t0 = time.perf_counter()
    ok = sum(fol_entails(x["premises-FOL"], x["conclusion-FOL"], time_limit=0.5, **kw) == (x["label"] == "True") for x in folio_valid)
    print(f"  {name:20s} {100*ok/len(folio_valid):6.2f}%   {time.perf_counter()-t0:6.2f}s")


Backward-chaining state-space search on RuleTaker test (accuracy / avg nodes expanded / time)
  bfs     97.60%     2.9 nodes    0.02s
  dfs     97.60%     3.3 nodes    0.02s
  ucs     97.60%     2.9 nodes    0.02s
  astar   97.60%     2.9 nodes    0.02s

FOLIO validation (logic given): resolution with different clause-selection strategies vs DPLL
  Resolution + A*       87.19%    11.01s
  Resolution + BFS      84.24%    14.92s
  Resolution + DFS      84.24%    16.09s
  DPLL (ground SAT)     87.68%     1.45s


## 6. Reasoning traces

In [42]:
# CELL 8: Example reasoning traces

# (a) Forward chaining with a proof tree (assignment example: variables + conjunction)
demo_ctx = """Bob is young.
Bob is kind.
If someone is young, then they are smart.
If someone is smart and kind, then they are successful."""
for q in ["Bob is successful.", "Bob is rich."]:
    ans, lines = predict(demo_ctx, q, trace=True)
    print(f"Query: {q}  ->  {ans}"); print("\n".join("   " + l for l in lines)); print()

# (b) First-order resolution refutation (Socrates), with the empty-clause derivation
prem = ["∀x (Man(x) → Mortal(x))", "Man(socrates)"]
concl = "Mortal(socrates)"
kb = [c for p in prem for c in to_clauses(parse_fol(p))]
goal = to_clauses(("not", parse_fol(concl)))
print("Clauses:", [fmt_clause(c) for c in kb], "| negated goal:", [fmt_clause(c) for c in goal])
P = Prover("astar"); tr = []
print("Resolution proves it:", P.refute(kb, goal, tr), "| clauses expanded:", P.stats["given"])
for a, b, why in tr: print(f"   resolve  {fmt_clause(a)}   with   {fmt_clause(b)}   ->   {why}")

# (c) Counterfactual (structural causal model): abduction -> action -> prediction
g = "We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune"
q = "Would Lumbo occur if not Ziklo instead of Ziklo?"
mech, priors, ev, factual, do, target = scm_parse(g, q)
print("\nMechanisms:", {k: [(op, l, n) for op, l, n in v] for k, v in mech.items()})
print("Observed:", ev, "| intervention do():", do, "| target:", target, "->", scm_answer(g, q))


Query: Bob is successful.  ->  True
   successful(bob)   [by rule: If someone is smart and kind, then they are successful.]
     smart(bob)   [by rule: If someone is young, then they are smart.]
       young(bob)   [given fact]
     kind(bob)   [given fact]

Query: Bob is rich.  ->  False
   rich(bob): not derivable -> False

Clauses: ['mortal(?v7795) v ~man(?v7795)', 'man(socrates)'] | negated goal: ['~mortal(socrates)']
Resolution proves it: True | clauses expanded: 2
   resolve  ~man(socrates)   with   man(socrates)   ->   contradiction (empty clause)

Mechanisms: {'Ziklo': [('and', [('Blaf', False)], False)], 'Trune': [('or', [('Ziklo', False), ('Blaf', False)], False)], 'Vork': [('and', [('Trune', False)], False)], 'Lumbo': [('and', [('Vork', False)], False)]}
Observed: {'Trune': 1} | intervention do(): {'Ziklo': 0} | target: Lumbo -> True


## 6. Generate `submission.csv` for the hidden test set



In [43]:
# CELL 9: Inference on the hidden test set -> submission/submission.csv

def load_table(path):
    p = str(path).lower()
    if p.endswith(".csv"):     return pd.read_csv(path)
    if p.endswith(".parquet"): return pd.read_parquet(path)
    if p.endswith(".jsonl"):   return pd.DataFrame(load_jsonl(path))
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return pd.DataFrame(data if isinstance(data, list) else data.get("data", data.get("examples", [])))

def pick(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

hidden_path = Path(v_HIDDEN_EVAL_FILE)
if hidden_path.exists():
    test_df = load_table(hidden_path)
    print("Loaded hidden test set:", hidden_path, test_df.shape)
else:
    test_df = ruletaker_test.copy()
    print(f"[!] {hidden_path} not found -> using the public RuleTaker test split to demo the output format.")

ctx_col  = pick(test_df, ["context", "premises", "given_info", "kb", "knowledge_base"])
qry_col  = pick(test_df, ["statement", "question", "query", "conclusion"])
pfol_col = pick(test_df, ["premises-FOL", "premises_fol", "premises_logic", "context_fol", "kb_fol", "logic_context"])
cfol_col = pick(test_df, ["conclusion-FOL", "conclusion_fol", "conclusion_logic", "query_fol", "statement_fol", "logic_query"])
id_col   = pick(test_df, ["id", "ID", "question_id", "example_id"])
assert (pfol_col and cfol_col) or (ctx_col and qry_col), f"Could not find input columns in {list(test_df.columns)}"

t0 = time.perf_counter()
labels = []
for _, r in test_df.iterrows():
    if pfol_col and cfol_col:
        labels.append(predict(premises_fol=r[pfol_col], conclusion_fol=r[cfol_col]))
    else:
        labels.append(predict(context=r[ctx_col], statement=r[qry_col]))
infer_time = time.perf_counter() - t0

submission = pd.DataFrame({"id": test_df[id_col] if id_col else range(len(test_df)),
                           "label": ["True" if l else "False" for l in labels]})
os.makedirs("submission", exist_ok=True)
submission.to_csv("submission/submission.csv", index=False)
print(f"Wrote submission/submission.csv  ({len(submission)} rows)")
print(f"Total inference time: {infer_time:.2f}s  ({1000*infer_time/len(submission):.2f} ms / query)")

gold_col = pick(test_df, ["flag", "label", "answer"])
if gold_col:
    gold = test_df[gold_col].astype(str).str.strip().str.lower().isin(["true", "yes", "1", "entailment"])
    print(f"Accuracy on this file: {100*(pd.Series(labels, index=test_df.index) == gold).mean():.2f}%")


[!] private_test.csv not found -> using the public RuleTaker test split to demo the output format.
Wrote submission/submission.csv  (250 rows)
Total inference time: 0.09s  (0.37 ms / query)
Accuracy on this file: 97.60%


## 7. Single Self-Contained Cell

In [45]:
# ==========================================================================================
#  SINGLE SELF-CONTAINED CELL  -  Classical Reasoning Agent  (EVALUATION CELL)

# ==========================================================================================
HIDDEN_TEST_LINK = "private_test.csv"
#   ^ put the PRIVATE TEST FILE here. Accepted:
#       - a local path          "private_test.csv", "data/test.parquet", "C:/files/test.json"
#       - a direct web link     "https://.../test.csv"
#       - share links           GitHub (blob), Hugging Face (blob), Google Drive, Dropbox
#     File types: csv / json / jsonl / parquet (detected from the content if the name has no extension)
SUBMISSION_FILE  = "submission/submission.csv"      # output file, columns: id,label
COLUMN_OVERRIDES = {}
#   ^ OPTIONAL. Leave empty: columns are detected automatically from their content.
#     Only if detection ever picks a wrong column, force it, e.g.
#     {"context": "premises", "statement": "conclusion", "id": "ID", "label": "label"}   (a name or a 0-based column number;
#     other keys: "premises_fol", "conclusion_fol")
# --------------------------------------------------------------------------------------------

import re, time, os, sys, json, csv, itertools, heapq, difflib, warnings, traceback, tempfile
import urllib.request, urllib.parse
from pathlib import Path
warnings.filterwarnings("ignore")
sys.setrecursionlimit(20000)


# ============================================================================================
#  RULE ENGINE: parser, forward chaining, backward chaining, counterfactual rules
# ============================================================================================
import re, time, os, sys, json, itertools

from collections import defaultdict, namedtuple

sys.setrecursionlimit(20000)



# ---------------- CONFIG ----------------

v_LOGIC_MODE = "cwa"                  # "cwa": closed-world, negation = failure (matches RuleTaker samples); "owa": explicit/strong negation only

v_INFERENCE_MODE = "fc"                       # "fc" (forward chaining) or "bc" (backward chaining)

v_HIDDEN_EVAL_FILE = "test_1_unicode.csv"   # path of the hidden test file (csv/json/jsonl/parquet) with context+question(+id) columns

v_RESULT_FILE = "submission.csv"

v_DEV_SAMPLE_LIMIT = 3000         # stratified (by config) sample size for dev evaluation; None = evaluate everything

# HuggingFace datasets: (short name, repo id, config). Wrong/unavailable repos are skipped automatically.

v_REMOTE_CORPORA = [                     # (name, repo, config, preferred eval split)  -- schemas checked: see section 5

    ("RuleTaker", "tasksource/ruletaker", None, "test"),   # columns: context, question, label(entailment|not entailment), config

    ("FOLIO", "yale-nlp/FOLIO", None, "validation"),         # columns: premises, conclusion, label(True|False|Uncertain), ...

    ("CounterBench", "CounterBench/CounterBench", None, None),  # binary yes/no counterfactual questions

]

v_RANDOM_STATE = 42

# Local files can also be evaluated: {"name": "path"}

v_EXTRA_EVAL_FILES = {}               # e.g. {"my_dev": "dev.csv"}

v_BACKUP_DATA_DIR = "data"              # bundled real records, used only if a Hugging Face download fails



v_Literal = namedtuple("Lit", "neg pred args")

v_InferenceRule = namedtuple("Rule", "body head text")

def v_is_variable(t): return isinstance(t, str) and t.startswith("?")



v_ARTICLES = {"the", "a", "an"}

v_PRONOUNS = {"they", "it", "he", "she", "them", "him", "her"}

v_QUANTIFIER_WORDS = {"someone", "something", "somebody", "anyone", "anything", "anybody", "everyone", "everything", "everybody"}

v_GENERIC_NOUNS = {"people", "things", "person", "thing", "ones", "persons", "individuals"}

v_COPULAR_FORMS = {"is", "are", "was", "were", "isn't", "aren't"}

v_AUXILIARY_FORMS = {"does", "do", "did", "doesn't", "don't"}

v_IRREGULAR_VERBS = {"has": "have", "does": "do", "goes": "go", "is": "be"}



def v_split_tokens(s):

    return re.findall(r"[A-Za-z0-9_'’-]+", s.replace("’", "'"))



def v_normalize_verb(w):

    if w in v_IRREGULAR_VERBS: return v_IRREGULAR_VERBS[w]

    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"

    if re.search(r"(ches|shes|sses|xes|zes)$", w): return w[:-2]

    if w.endswith("s") and not w.endswith("ss"): return w[:-1]

    return w



def v_singularize(w):

    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"

    if w.endswith("s") and not w.endswith("ss") and len(w) > 3: return w[:-1]

    return w



def v_entity_term(tokens, role="subj"):

    toks = [t for t in tokens if t.lower() not in v_ARTICLES]

    if not toks: return None

    if len(toks) == 1:

        w = toks[0].lower()

        if w in v_PRONOUNS: return "?x"

        if w in v_QUANTIFIER_WORDS: return "?x" if role == "subj" else "?y"

    return "_".join(t.lower() for t in toks)



def v_predicate_name(toks):

    toks = [t.lower() for t in toks]

    while toks and toks[0] in v_ARTICLES: toks = toks[1:]

    return "_".join(toks) if toks else None



def v_locate_predicate(toks, allow_base=False):

    low = [t.lower() for t in toks]; n = len(toks)

    for i in range(1, n):

        w = low[i]

        if w in v_COPULAR_FORMS: return i, "cop"

        if w in v_AUXILIARY_FORMS: return i, "aux"

        if (i + 1 < n and w.endswith("s") and w != "is" and

                (low[i + 1] in v_ARTICLES or toks[i + 1][0].isupper() or low[i + 1] in v_PRONOUNS | v_QUANTIFIER_WORDS)):

            return i, "verb"

    if low[0] in v_PRONOUNS | v_QUANTIFIER_WORDS and n > 1: return 1, "verb_base"

    if allow_base and n > 1: return 1, "verb_base"

    return None



def v_parse_clause(toks, allow_base=False):

    sp = v_locate_predicate(toks, allow_base)

    if not sp: return None

    i, kind = sp

    subj = v_entity_term(toks[:i], "subj")

    if subj is None: return None

    low = [t.lower() for t in toks]

    if kind == "cop":

        neg = low[i] in ("isn't", "aren't")

        rest = toks[i + 1:]

        if rest and rest[0].lower() == "not": neg, rest = True, rest[1:]

        p = v_predicate_name(rest)

        if not p: return None

        return [v_Literal(neg, p, (subj,))], ("cop", subj)

    neg = False

    if kind == "aux":

        neg = low[i] in ("doesn't", "don't")

        j = i + 1

        if j < len(toks) and low[j] == "not": neg, j = True, j + 1

        if j >= len(toks): return None

        verb, rest = low[j], toks[j + 1:]

    elif kind == "verb":

        verb, rest = v_normalize_verb(low[i]), toks[i + 1:]

    else:

        verb, rest = low[i], toks[i + 1:]

        if rest and rest[0].lower() == "not": neg, rest = True, rest[1:]

    obj = v_entity_term(rest, "obj") if rest else None

    args = (subj,) if obj is None else (subj, obj)

    return [v_Literal(neg, verb, args)], ("verb", subj, verb, neg)



def v_parse_continuation(toks, state):

    if state[0] == "cop":

        neg = False

        if toks and toks[0].lower() == "not": neg, toks = True, toks[1:]

        p = v_predicate_name(toks)

        return [v_Literal(neg, p, (state[1],))] if p else None

    obj = v_entity_term(toks, "obj")

    return [v_Literal(state[3], state[2], (state[1], obj))] if obj else None



def v_fallback_proposition(toks):

    low = [t.lower() for t in toks if t.lower() not in v_ARTICLES]

    neg = False

    if low and low[0] == "not": neg, low = True, low[1:]

    return [v_Literal(neg, "prop_" + "_".join(low), ())]



def v_parse_conjunction(text, allow_base=False):

    pieces = [p for p in re.split(r"\s*,\s*and\s+|\s+and\s+|\s*,\s*", text.strip()) if p.strip()]

    lits, state = [], None

    for p in pieces:

        toks = v_split_tokens(p)

        if not toks: continue

        full = v_parse_clause(toks, allow_base and state is None)

        if full: l, state = full; lits += l; continue

        if state:

            c = v_parse_continuation(toks, state)

            if c: lits += c; continue

        lits += v_fallback_proposition(toks)

    return lits



def v_contains_variable(lits): return any(v_is_variable(a) for l in lits for a in l.args)



def v_parse_statement(s):

    """Returns ('fact', [Lit]) or ('rule', [Rule])."""

    orig = s.strip(); s = orig.rstrip(".!?").strip()

    if not s: return None

    m = re.match(r"^(?:if|when|whenever)\s+(.*?)\s*,?\s+then\s+(.*)$", s, re.I) or \
        re.match(r"^(?:if|when|whenever)\s+([^,]*),\s*(.*)$", s, re.I)

    if m:

        body, head = v_parse_conjunction(m.group(1)), v_parse_conjunction(m.group(2))

        return "rule", [v_InferenceRule(tuple(body), h, orig) for h in head]

    toks = v_split_tokens(s); low = [t.lower() for t in toks]

    # universal statements: "All/Every X are Y", "Big, red things are nice", "Young people are kind"

    subj_end = None; body_start = 0; plural_noun = False

    if low and low[0] in ("all", "every", "each", "any") and len(toks) > 2:

        sp = v_locate_predicate(toks[1:], allow_base=True)

        if sp: subj_end, body_start, plural_noun = sp[0] + 1, 1, True

    else:

        for g, w in enumerate(low):

            if w in v_COPULAR_FORMS or w in v_AUXILIARY_FORMS: break

            if w in v_GENERIC_NOUNS and g < len(toks) - 1: subj_end = g + 1; break

    if subj_end:

        subj_toks = toks[body_start:subj_end]

        if subj_toks and subj_toks[-1].lower() in v_GENERIC_NOUNS: subj_toks = subj_toks[:-1]

        elif plural_noun and subj_toks: subj_toks = subj_toks[:-1] + [v_singularize(subj_toks[-1].lower())]

        body, neg = [], False

        for t in subj_toks:                     # RuleTaker adjectives are single words ("Big, red things ...")

            w = t.lower()

            if w == "and" or w in v_ARTICLES: continue

            if w == "not": neg = True; continue

            body.append(v_Literal(neg, w, ("?x",))); neg = False

        head = v_parse_conjunction("they " + " ".join(toks[subj_end:]))

        return "rule", [v_InferenceRule(tuple(body), h, orig) for h in head]

    lits = v_parse_conjunction(s)

    if v_contains_variable(lits): return "rule", [v_InferenceRule((), h, orig) for h in lits]   # e.g. "Everything is big"

    return "fact", lits



def v_segment_sentences(context):

    if isinstance(context, (list, tuple)): context = "\n".join(map(str, context))

    out = []

    for line in str(context).split("\n"):

        for s in re.split(r"(?<=[.!?])\s+", line.strip()):

            if s.strip(): out.append(s.strip())

    return out



def v_parse_knowledge_context(context):

    facts, rules = [], []

    for s in v_segment_sentences(context):

        r = v_parse_statement(s)

        if not r: continue

        (facts.extend if r[0] == "fact" else rules.extend)(r[1])

    return facts, rules



def v_parse_question(q):

    q = re.sub(r"^\s*(query|question|statement|hypothesis|conclusion)\s*:\s*", "", str(q), flags=re.I)

    q = re.sub(r"^\s*(true or false|is it true that|is the following true)\s*[:?]?\s*", "", q, flags=re.I)

    return v_parse_conjunction(q.strip().rstrip(".!?"))



def v_format_atom(pred, args): return f"{pred}({', '.join(a.replace('_',' ') for a in args)})" if args else pred



def v_reduce_knowledge_base(facts, rules, goal_preds):

    deps = defaultdict(set)

    for r in rules: deps[r.head.pred] |= {l.pred for l in r.body}

    rel, stack = set(goal_preds), list(goal_preds)

    while stack:

        p = stack.pop()

        for q in deps.get(p, ()):

            if q not in rel: rel.add(q); stack.append(q)

    return [f for f in facts if f[0] in rel], [r for r in rules if r.head.pred in rel]



def v_compute_strata(rules):

    preds = set()

    for r in rules: preds.add(r.head.pred); preds |= {l.pred for l in r.body}

    s = {p: 0 for p in preds}

    for _ in range(len(preds) + 1):

        ch = False

        for r in rules:

            for l in r.body:

                need = s[l.pred] + (1 if l.neg else 0)

                if s[r.head.pred] < need: s[r.head.pred] = need; ch = True

        if not ch: break

    return s



class v_ForwardInference:

    def __init__(self, facts, rules, domain=()):

        self.db = defaultdict(set); self.idx = defaultdict(set); self.prov = {}

        self.rules = list(rules)

        dom = set(domain)

        for p, a in facts:

            self.v__add(p, a, None); dom |= set(a)

        for r in rules:

            for l in list(r.body) + [r.head]: dom |= {a for a in l.args if not v_is_variable(a)}

        self.domain = sorted(dom)

        self.compiled = [self.v__compile(i, r) for i, r in enumerate(rules)]



    def v__add(self, pred, args, prov):

        if args in self.db[pred]: return False

        self.db[pred].add(args)

        for j, a in enumerate(args): self.idx[(pred, j, a)].add(args)

        if prov is not None: self.prov[(pred, args)] = prov

        return True



    def v__compile(self, i, r):

        pos = [l for l in r.body if not l.neg]; neg = [l for l in r.body if l.neg]

        order, bound = [], set()

        while pos:

            def score(l):

                b = sum(1 for a in l.args if (not v_is_variable(a)) or a in bound)

                return (-b, len(self.db.get(l.pred, ())))

            nxt = min(pos, key=score); pos.remove(nxt); order.append(nxt)

            bound |= {a for a in nxt.args if v_is_variable(a)}

        return (i, order, neg, r.head)



    def v__cands(self, pred, args):

        best = None

        for j, a in enumerate(args):

            if not v_is_variable(a):

                s = self.idx.get((pred, j, a))

                if s is None: return ()

                if best is None or len(s) < len(best): best = s

        return best if best is not None else self.db.get(pred, ())



    def v__solve(self, lits, i, th):

        if i == len(lits): yield th; return

        l = lits[i]

        args = tuple(th.get(a, a) if v_is_variable(a) else a for a in l.args)

        for tup in self.v__cands(l.pred, args):

            new, ok = th, True

            for a, v in zip(args, tup):

                if v_is_variable(a):

                    if new is th: new = dict(th)

                    if a in new:

                        if new[a] != v: ok = False; break

                    else: new[a] = v

                elif a != v: ok = False; break

            if ok: yield from self.v__solve(lits, i + 1, new)



    def v__negs_ok(self, negs, th):

        for l in negs:

            args = tuple(th.get(a, a) if v_is_variable(a) else a for a in l.args)

            if any(v_is_variable(a) for a in args):

                if any(True for _ in self.v__solve([v_Literal(False, l.pred, args)], 0, {})): return False

            elif args in self.db.get(l.pred, ()): return False

        return True



    def v__fire(self, cr):

        i, pos, neg, head = cr

        derived = []

        nvars = sorted({a for l in neg for a in l.args if v_is_variable(a)})

        for th0 in self.v__solve(pos, 0, {}):

            free_n = [v for v in nvars if v not in th0]          # variables bound only by negative literals -> range over domain

            for combo in (itertools.product(self.domain, repeat=len(free_n)) if free_n else [()]):

                th = {*th0, *dict(zip(free_n, combo))} if free_n else th0

                if neg and not self.v__negs_ok(neg, th): continue

                self.v__emit(th, pos, neg, head, i, derived)

        c = 0

        for p, a, pv in derived: c += self.v__add(p, a, pv)

        return c



    def v__emit(self, th, pos, neg, head, i, derived):

        if True:

            hargs = tuple(th.get(a, a) if v_is_variable(a) else a for a in head.args)

            free = sorted({a for a in hargs if v_is_variable(a)})

            combos = [dict(zip(free, c)) for c in itertools.product(self.domain, repeat=len(free))] if free else [{}]

            for m in combos:

                ga = tuple(m.get(a, a) for a in hargs)

                if ga in self.db.get(head.pred, ()): continue

                sup = [(l.pred, tuple(th.get(a, a) if v_is_variable(a) else a for a in l.args)) for l in pos]

                nsup = [(l.pred, tuple(th.get(a, a) if v_is_variable(a) else a for a in l.args)) for l in neg]

                derived.append((head.pred, ga, (i, sup, nsup)))



    def v_run(self, goal=None):

        st = v_compute_strata(self.rules)

        groups = defaultdict(list)

        for cr in self.compiled: groups[st.get(cr[3].pred, 0)].append(cr)

        for k in sorted(groups):

            while True:

                n = sum(self.v__fire(cr) for cr in groups[k])

                if goal and goal[1] in self.db.get(goal[0], ()): return

                if not n: break



    def v_holds(self, lit):

        present = lit.args in self.db.get(lit.pred, ())

        return (not present) if lit.neg else present



    def v_explain(self, fact, indent=0, seen=None):

        seen = set() if seen is None else seen

        pad = "  " * indent; pred, args = fact

        pv = self.prov.get(fact)

        if pv is None: return [f"{pad}{v_format_atom(pred,args)}   [given fact]"]

        if fact in seen: return [f"{pad}{v_format_atom(pred,args)}   [see above]"]

        seen.add(fact)

        i, sup, nsup = pv

        out = [f"{pad}{v_format_atom(pred,args)}   [by rule: {self.rules[i].text}]"]

        for s in sup: out += self.v_explain(s, indent + 1, seen)

        for p, a in nsup: out.append(f"{pad}  not {v_format_atom(p,a)}   [negation as failure: not derivable]")

        return out



class v_BackwardInference:

    def __init__(self, facts, rules, max_depth=30):

        self.facts = defaultdict(set); self.rules = defaultdict(list)

        for p, a in facts: self.facts[p].add(a)

        for r in rules: self.rules[r.head.pred].append(r)

        self.max_depth = max_depth; self.cnt = 0

    def v_walk(self, t, th):

        while v_is_variable(t) and t in th: t = th[t]

        return t

    def v_unify(self, a, b, th):

        if len(a) != len(b): return None

        th = dict(th)

        for x, y in zip(a, b):

            x, y = self.v_walk(x, th), self.v_walk(y, th)

            if x == y: continue

            if v_is_variable(x): th[x] = y

            elif v_is_variable(y): th[y] = x

            else: return None

        return th

    def v_rename(self, r):

        self.cnt += 1; s = f"#{self.cnt}"

        f = lambda a: tuple(x + s if v_is_variable(x) else x for x in a)

        body = [v_Literal(l.neg, l.pred, f(l.args)) for l in r.body]

        body = [l for l in body if not l.neg] + [l for l in body if l.neg]   # NAF goals last

        return body, v_Literal(r.head.neg, r.head.pred, f(r.head.args))

    def v_solve(self, goals, th, depth):

        if not goals: yield th; return

        (g, stack), rest = goals[0], goals[1:]

        args = tuple(self.v_walk(a, th) for a in g.args)

        if g.neg:

            if not any(True for _ in self.v_solve(((v_Literal(False, g.pred, args), frozenset()),), th, depth + 1)):

                yield from self.v_solve(rest, th, depth)

            return

        key = (g.pred, args)

        if key in stack or depth > self.max_depth: return

        st2 = stack | {key}

        for tup in self.facts.get(g.pred, ()):

            t2 = self.v_unify(args, tup, th)

            if t2 is not None: yield from self.v_solve(rest, t2, depth)

        for r in self.rules.get(g.pred, ()):

            body, head = self.v_rename(r)

            t2 = self.v_unify(args, head.args, th)

            if t2 is not None: yield from self.v_solve(tuple((b, st2) for b in body) + rest, t2, depth + 1)

    def v_prove(self, lits):

        return any(True for _ in self.v_solve(tuple((l, frozenset()) for l in lits), {}, 0))



def v__cb_lit(txt):

    t = txt.strip().strip(".?,").strip()

    neg = False

    if t.lower().startswith("not "): neg, t = True, t[4:].strip()

    return neg, t



def v_is_counterfactual_prompt(text):

    return bool(re.search(r"\bcauses?\b", text)) and bool(re.search(r"\bwould\b.*\boccur\b", text, re.I | re.S))



def v_parse_counterfactual_prompt(text):

    """Returns (rules, facts, target_name, interventions[(name,value)], domain) or None if the text is not parsable."""

    m = re.search(r"We know that\s+(.*?)\.(?=\s|$)", text, re.S)

    if not m: return None

    segs, stm = [x.strip() for x in re.split(r",\s*", m.group(1)) if x.strip()], []

    for sg in segs:

        sg = re.sub(r"^and\s+", "", sg)

        if re.search(r"\bcauses?\b", sg) or not stm: stm.append(sg)

        else: stm[-1] += ", " + sg                       # continuation of a target list: "X causes A, B and C"

    rules, domain, targets = [], set(), set()

    for st in stm:

        mm = re.match(r"(.*?)\s+causes?\s+(.*)$", st)

        if not mm: return None

        lhs = re.sub(r"\s+together$", "", mm.group(1).strip())

        dnf = [[v__cb_lit(l) for l in re.split(r"\s+and\s+", clause)] for clause in re.split(r"\s+or\s+", lhs)]

        for tg in re.split(r"\s*,\s*|\s+and\s+", mm.group(2)):

            if not tg.strip(): continue

            tneg, tname = v__cb_lit(tg); targets.add(tname); domain.add(tname)

            d = dnf

            if tneg:                                       # De Morgan: NOT(OR_i AND_j l_ij) = AND_i OR_j NOT l_ij  -> DNF by product

                d = [list(c) for c in itertools.product(*[[(not n, v) for n, v in cl] for cl in dnf])]

            for cl in d:

                for n, v in cl: domain.add(v)

                rules.append(v_InferenceRule(tuple(v_Literal(n, "occurs", (v,)) for n, v in cl), v_Literal(False, "occurs", (tname,)), st))

    mt = re.search(r"[Ww]ould\s+(.*?)\s+occur", text)

    if not mt or not rules: return None

    target = v__cb_lit(mt.group(1))[1]; domain.add(target)

    interventions = []

    mi = re.search(r"occur\s+if\s+(.*?)(?:\s+instead of\b.*)?\?", text, re.S)

    if mi:

        for l in re.split(r"\s+and\s+", mi.group(1)):

            n, v = v__cb_lit(l); interventions.append((v, 0 if n else 1))

    for kw in ("[Aa]ssume", "suppose"):

        for mm in re.finditer(kw + r"\s+((?:not\s+)?[^,.?]+)", text):

            n, v = v__cb_lit(mm.group(1)); interventions.append((v, 0 if n else 1))

    facts = set()

    for mm in re.finditer(r"[Ww]e observed\s+([^.?]+)", text):

        for l in re.split(r"\s+and\s+|,\s*", mm.group(1)):

            n, v = v__cb_lit(l)

            if not n: facts.add(("occurs", (v,)))

            domain.add(v)

    return rules, facts, target, interventions, domain



def v_answer_counterfactual(text, trace=False):

    p = v_parse_counterfactual_prompt(text)

    if p is None: return None

    rules, facts, target, ints, domain = p

    for v, val in ints:

        rules = [r for r in rules if r.head.args[0] != v]            # graph surgery: cut all incoming edges of v

        facts.discard(("occurs", (v,)))

        if val == 1: facts.add(("occurs", (v,)))

    facts, rules2 = v_reduce_knowledge_base(facts, rules, ["occurs"])

    fc = v_ForwardInference(facts, rules2, domain); fc.v_run()

    res = (target,) in fc.db.get("occurs", ())

    if not trace: return res

    lines = [f"intervention do({v}={val})" for v, val in ints]

    lines += fc.v_explain(("occurs", (target,))) if res else [f"occurs({target}) is not derivable -> No"]

    return res, lines



def v_construct_knowledge_base(context, semantics=None):

    semantics = semantics or v_LOGIC_MODE

    fl, rules = v_parse_knowledge_context(context)

    domain = {c for l in fl for c in l.args} | {c for r in rules for l in list(r.body) + [r.head] for c in l.args if not v_is_variable(c)}

    if semantics == "owa":

        tr = lambda l: v_Literal(False, "~" + l.pred, l.args) if l.neg else l      # strong negation as separate atom ~p

    else:

        # closed world: "not p" in bodies/queries = negation as failure. Explicit negative facts and negative rule

        # heads are redundant (a consistent KB never derives p when it asserts not-p), so they are dropped.

        fl = [l for l in fl if not l.neg]; rules = [r for r in rules if not r.head.neg]

        tr = lambda l: l

    facts = {(l.pred, l.args) for l in map(tr, fl)}

    rules = [v_InferenceRule(tuple(tr(b) for b in r.body), tr(r.head), r.text) for r in rules]

    return facts, rules, tr, domain



def v_infer_answer(context, query, engine=None, trace=False, semantics=None):

    engine = engine or v_INFERENCE_MODE

    full = f"{context} {query}"

    if v_is_counterfactual_prompt(full):

        r = v_answer_counterfactual(full, trace)

        if r is not None: return r

    facts, rules, tr, domain = v_construct_knowledge_base(context, semantics)

    ql = v_parse_question(query)

    domain |= {c for l in ql for c in l.args if not v_is_variable(c)}

    q = [tr(l) for l in ql]

    if not q: return (False, ["(could not parse query)"]) if trace else False

    facts, rules = v_reduce_knowledge_base(facts, rules, [l.pred for l in q])

    if engine == "bc":

        res = v_BackwardInference(facts, rules).v_prove(q)

        return (res, ["(backward chaining; no trace)"]) if trace else res

    fc = v_ForwardInference(facts, rules, domain)

    goal = (q[0].pred, q[0].args) if len(q) == 1 and not q[0].neg else None

    fc.v_run(goal)

    res = all(fc.v_holds(l) for l in q)

    if not trace: return res

    lines = []

    for l in q:

        if l.neg: lines.append(f"not {v_format_atom(l.pred,l.args)}: {'holds by negation as failure' if fc.v_holds(l) else 'FAILS (it is derivable)'}")

        elif fc.v_holds(l): lines += fc.v_explain((l.pred, l.args))

        else: lines.append(f"{v_format_atom(l.pred,l.args)}: not derivable -> False")

    return res, lines



def v_evaluate_counterfactuals(path="datasets/counterbench/data_balanced_alpha_V1.json"):

    with open(path, "r", encoding="utf-8") as f:

        data = json.load(f)

    if isinstance(data, dict):

        data = data.get("data", data.get("examples", []))

    total = correct = 0

    by_type = defaultdict(lambda: [0, 0])

    rows = []

    errors = []

    for ex in data:

        question = ex.get("question", ex.get("query", ""))

        context = ex.get("given_info", ex.get("context", ""))

        gold_value = str(ex.get("answer", "")).strip().lower()

        if gold_value not in {"yes", "no", "true", "false", "1", "0"}:

            continue

        gold = gold_value in {"yes", "true", "1"}

        error_message = ""

        try:

            pred = v_infer_answer(context, question)

            if isinstance(pred, tuple):

                pred = pred[0]

            pred = bool(pred)

        except Exception as error:

            # Preserve the original fallback prediction, but retain the reason for diagnosis.

            pred = False

            error_message = f"{type(error).__name__}: {error}"

        ok = pred == gold

        total += 1

        correct += int(ok)

        typ = str(ex.get("type", "unknown"))

        by_type[typ][0] += int(ok)

        by_type[typ][1] += 1

        row = {

            "question_id": ex.get("question_id", ex.get("id", total)),

            "type": typ,

            "question": question,

            "gold": "yes" if gold else "no",

            "prediction": "yes" if pred else "no",

            "correct": ok,

            "error": error_message

        }

        rows.append(row)

        if not ok:

            errors.append({

                **row,

                "context": context

            })

    print(f"CounterBench accuracy: {correct}/{total} = {100*correct/max(total,1):.2f}%")

    for typ, (ncorrect, n) in sorted(by_type.items()):

        print(f"{typ}: {ncorrect}/{n} = {100*ncorrect/max(n,1):.2f}%")

    os.makedirs("submission", exist_ok=True)

    import csv

    with open("submission/counterbench_predictions.csv", "w", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else [])

        if rows:

            writer.writeheader()

            writer.writerows(rows)

    print("Saved submission/counterbench_predictions.csv")

    error_path = os.path.join("submission", "counterbench_errors.csv")

    with open(error_path, "w", newline="", encoding="utf-8") as f:

        fields = ["question_id", "type", "question", "context", "gold", "prediction", "correct", "error"]

        writer = csv.DictWriter(f, fieldnames=fields)

        writer.writeheader()

        writer.writerows(errors)

    print(f"Saved {error_path} ({len(errors)} incorrect predictions)")

    return 100*correct/max(total,1)



# (CounterBench evaluation now runs in CELL 7; call v_evaluate_counterfactuals() manually only if you want the per-example CSVs)


# ---- re-apply the user settings (the engine code above has its own defaults) ----
v_HIDDEN_EVAL_FILE = HIDDEN_TEST_LINK


# ============================================================================================
#  BACKWARD-CHAINING PROOF SEARCH: BFS / DFS / UCS / A*
# ============================================================================================
import heapq, itertools, re

def horn_from_ruletaker(context):
    """Parse a (normalised) RuleTaker context into ground facts and single-premise Horn rules over string atoms."""
    T = re.compile(r"^(?:given that|based on|assuming|when|if)\s+(.*?),\s*(?:it follows that|it is evident that|we can say|we can conclude|then)\s+(.*?)\.?\s*$", re.I)
    norm = lambda x: re.sub(r"\s+", " ", re.sub(r"^the\s+", "", x.lower().strip().rstrip(".")))
    facts, rules = set(), []
    for l in str(context).split("\n"):
        l = l.strip()
        if not l or l.lower() in ("facts:", "rules:"): continue
        m = T.match(l)
        if m: rules.append((norm(m.group(1)), norm(m.group(2))))
        else: facts.add(norm(l))
    return facts, rules

def prove(goal, facts, rules, strategy="astar", max_nodes=20000):
    """Backward-chaining proof search as a state-space search.
    state = frozenset of open sub-goals; a move replaces one goal by the body of a rule whose head it is
    (or discards it if it is a known fact). Goal state = no open sub-goals.
      bfs   : FIFO frontier            dfs : LIFO frontier
      ucs   : priority = g (rules used)    astar : priority = g + h,  h = number of open sub-goals."""
    by_head = {}
    for b, h in rules: by_head.setdefault(h, []).append(b)
    start = frozenset([goal]); tick = itertools.count(); seen = {start}; nodes = 0
    frontier = [(0, next(tick), 0, start)] if strategy in ("ucs", "astar") else None
    queue = [(0, start)]
    while (frontier if frontier is not None else queue):
        if frontier is not None: _, _, g, st = heapq.heappop(frontier)
        elif strategy == "bfs": g, st = queue.pop(0)
        else: g, st = queue.pop()
        nodes += 1
        if nodes > max_nodes: return False, nodes
        if not st: return True, nodes
        goal_ = sorted(st)[0]; rest = st - {goal_}
        nxt = []
        if goal_ in facts: nxt.append(rest)
        for body in by_head.get(goal_, []): nxt.append(rest | {body})
        for n in nxt:
            if n in seen: continue
            seen.add(n)
            if frontier is not None:
                h = len(n) if strategy == "astar" else 0
                heapq.heappush(frontier, (g + 1 + h, next(tick), g + 1, n))
            else: queue.append((g + 1, n))
    return False, nodes


# ============================================================================================
#  FIRST-ORDER LOGIC: parser, CNF, unification, resolution (BFS/DFS/A*), DPLL
# ============================================================================================
import re, time, itertools, heapq
from collections import defaultdict

# =====================================================================
# 1. FOL parser  (∀ ∃ ¬ ∧ ∨ ⊕ → ↔ = ≠, predicates, functions)
# =====================================================================
_TOK = re.compile(r"∀|∃|¬|∧|∨|⊕|→|↔|⟷|≠|=|\(|\)|,|[A-Za-z0-9_][A-Za-z0-9_'\-\.]*")
_QUANT = {"∀": "all", "∃": "ex"}

class FOLSyntaxError(Exception): pass
EXACTLY_ONE = True

def normalise_syntax(s):
    """Translate common ASCII / textbook notation into the Unicode symbols the parser uses."""
    s = s.replace("\u2014", "-")
    s = re.sub(r"\b(?:forall|for\s+all)\s+([A-Za-z]\w*(?:\s*,\s*[A-Za-z]\w*)*)\s*[.:]?\s*", lambda m: "∀" + re.sub(r"\s*,\s*", " ", m.group(1)) + " ", s, flags=re.I)
    s = re.sub(r"\b(?:exists|there\s+exists)\s+([A-Za-z]\w*(?:\s*,\s*[A-Za-z]\w*)*)\s*[.:]?\s*", lambda m: "∃" + re.sub(r"\s*,\s*", " ", m.group(1)) + " ", s, flags=re.I)
    for a, b in [("<->", "↔"), ("<=>", "↔"), ("->", "→"), ("=>", "→"), ("!=", "≠"), ("&&", "∧"), ("||", "∨")]:
        s = s.replace(a, b)
    s = re.sub(r"\bXOR\b", "⊕", s); s = re.sub(r"\bAND\b", "∧", s); s = re.sub(r"\bOR\b", "∨", s)
    s = re.sub(r"\bNOT\b", "¬ ", s); s = re.sub(r"\bIMPLIES\b", "→", s)
    return s.replace("&", "∧").replace("|", "∨").replace("^", "∧").replace("~", "¬").replace("!", "¬")

def _tokenize(s):
    return _TOK.findall(normalise_syntax(s).strip())

class _Parser:
    def __init__(self, text):
        self.t = _tokenize(text); self.i = 0; self.bound = []; self.free = set(); self.fuzzy_free = True
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        tok = self.peek()
        if tok is None or (x is not None and tok != x): raise FOLSyntaxError(f"expected {x} got {tok}")
        self.i += 1; return tok
    def parse(self):
        f = self.iff()
        if self.peek() is not None: raise FOLSyntaxError("trailing tokens")
        for v in sorted(self.free): f = ("all", v, f)          # Prover9 convention: free variables are universal
        return f
    def iff(self):
        a = self.imp()
        while self.peek() in ("↔", "⟷"):
            self.eat(); a = ("iff", a, self.imp())
        return a
    def imp(self):
        a = self.orx()
        if self.peek() == "→":
            self.eat(); return ("imp", a, self.imp())
        return a
    def orx(self):
        a = self.andx(); xs = [a]; only_xor = True
        while self.peek() in ("∨", "⊕"):
            op = self.eat(); b = self.andx()
            if op == "∨": a = ("or", a, b); only_xor = False; xs = [a]
            else:
                xs.append(b); a = ("xor", a, b)
        if only_xor and len(xs) > 2 and EXACTLY_ONE:            # a ⊕ b ⊕ c ... read as "exactly one of"
            atleast = xs[0]
            for y in xs[1:]: atleast = ("or", atleast, y)
            f = atleast
            for i in range(len(xs)):
                for j in range(i + 1, len(xs)):
                    f = ("and", f, ("or", ("not", xs[i]), ("not", xs[j])))
            return f
        return a
    def andx(self):
        a = self.unary()
        while self.peek() == "∧":
            self.eat(); a = ("and", a, self.unary())
        return a
    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.eat(); return ("not", self.unary())
        if tok in _QUANT:
            self.eat(); vs = []
            while self.peek() is not None and re.match(r"[A-Za-z_]", self.peek()) and self.peek() not in _QUANT:
                nxt = self.t[self.i + 1] if self.i + 1 < len(self.t) else None
                if vs and nxt == "(":   # a later identifier followed by "(" starts the body (a predicate)
                    break
                vs.append(self.eat())
                if self.peek() == ",": self.eat()
            if not vs: raise FOLSyntaxError("quantifier without variable")
            self.bound.extend(v.lower() for v in vs)
            body = self.unary()
            for _ in vs: self.bound.pop()
            for v in reversed(vs): body = (_QUANT[tok], v.lower(), body)
            return body
        if tok == "(":
            self.eat(); f = self.iff(); self.eat(")")
            # parenthesised term followed by = ? (rare) -> not supported
            return f
        return self.atom()
    def term(self):
        name = self.eat()
        if not re.match(r"[A-Za-z0-9_]", name): raise FOLSyntaxError("bad term " + name)
        low = name.lower()
        if self.peek() == "(":
            self.eat(); args = [self.term()]
            while self.peek() == ",": self.eat(); args.append(self.term())
            self.eat(")"); return (low,) + tuple(args)
        if low in self.bound: return "?" + low
        if self.fuzzy_free and re.fullmatch(r"[u-z]\d*", low): self.free.add(low); return "?" + low
        return low
    def atom(self):
        t = self.term()
        if self.peek() in ("=", "≠"):
            op = self.eat(); u = self.term(); a = ("atom", "eq", (t, u))
            return a if op == "=" else ("not", a)
        if isinstance(t, tuple): return ("atom", t[0], t[1:])
        if t.startswith("?"): raise FOLSyntaxError("variable used as proposition")
        return ("atom", t, ())

def parse_fol(text):
    return _Parser(text).parse()

def _balance(tokens):
    d = 0
    for t in tokens:
        d += (t == "(") - (t == ")")
    return d

def _try_parse_tokens(tokens):
    p = _Parser(""); p.t = tokens
    try: return p.parse()
    except (FOLSyntaxError, RecursionError, IndexError): return None

def _score(f):
    """prefer repairs that give the usual shape  forall x (A -> B)  /  A -> B  at the top."""
    while f[0] in ("all", "ex"): f = f[2]
    return 0 if f[0] in ("imp", "iff") else 1

def parse_fol_tolerant(text):
    """Parse; if parentheses are unbalanced, try the most plausible repair (the gold FOL contains such typos)."""
    toks = _tokenize(text)
    f = _try_parse_tokens(toks)
    if f is not None: return f
    d = _balance(toks)
    cands = []
    if d > 0:      # missing ")" : insert at every position from the end backwards
        for pos in range(len(toks), 0, -1):
            cands.append(toks[:pos] + [")"] + toks[pos:])
        extra = d - 1
        if extra > 0:
            cands = [c + [")"] * extra for c in cands]
    elif d < 0:    # surplus ")" : drop one from the right
        idx = [i for i, t in enumerate(toks) if t == ")"]
        for i in reversed(idx):
            c = toks[:i] + toks[i + 1:]
            for _ in range(-d - 1):
                j = max((k for k, t in enumerate(c) if t == ")"), default=None)
                if j is not None: c = c[:j] + c[j + 1:]
            cands.append(c)
    best = None
    for c in cands:
        g = _try_parse_tokens(c)
        if g is None: continue
        sc = _score(g)
        if best is None or sc < best[0]: best = (sc, g)
        if sc == 0: break
    if best: return best[1]
    # last resort: trailing junk
    for cut in range(len(toks) - 1, 2, -1):
        g = _try_parse_tokens(toks[:cut] + [")"] * max(0, _balance(toks[:cut])))
        if g is not None: return g
    raise FOLSyntaxError("unparsable")

# =====================================================================
# 2. Clause normal form (eliminate -> <-> xor, NNF, Skolemise, distribute)
# =====================================================================
_sk = itertools.count(1); _vr = itertools.count(1)

def _elim(f):
    k = f[0]
    if k == "atom": return f
    if k == "not": return ("not", _elim(f[1]))
    if k in ("and", "or"): return (k, _elim(f[1]), _elim(f[2]))
    if k == "imp": return ("or", ("not", _elim(f[1])), _elim(f[2]))
    if k == "iff":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", ("not", a), b), ("or", ("not", b), a))
    if k == "xor":
        a, b = _elim(f[1]), _elim(f[2])
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (k, f[1], _elim(f[2]))

def _nnf(f, neg=False):
    k = f[0]
    if k == "atom": return ("not", f) if neg else f
    if k == "not": return _nnf(f[1], not neg)
    if k in ("and", "or"):
        kk = {"and": "or", "or": "and"}[k] if neg else k
        return (kk, _nnf(f[1], neg), _nnf(f[2], neg))
    kk = {"all": "ex", "ex": "all"}[k] if neg else k
    return (kk, f[1], _nnf(f[2], neg))

def _subst_term(t, m):
    if isinstance(t, tuple): return (t[0],) + tuple(_subst_term(a, m) for a in t[1:])
    return m.get(t, t)

def _skolem(f, env, univ):
    k = f[0]
    if k == "atom": return ("atom", f[1], tuple(_subst_term(a, env) for a in f[2]))
    if k == "not": return ("not", _skolem(f[1], env, univ))
    if k in ("and", "or"): return (k, _skolem(f[1], env, univ), _skolem(f[2], env, univ))
    v = "?" + f[1]
    if k == "all":
        nv = f"?v{next(_vr)}"; e = dict(env); e[v] = nv
        return _skolem(f[2], e, univ + [nv])
    e = dict(env)
    e[v] = (f"sk{next(_sk)}",) + tuple(univ) if univ else f"sk{next(_sk)}"
    return _skolem(f[2], e, univ)

def _cnf(f):
    k = f[0]
    if k == "and": return _cnf(f[1]) + _cnf(f[2])
    if k == "or":
        a, b = _cnf(f[1]), _cnf(f[2])
        if len(a) * len(b) > 4000: raise FOLSyntaxError("CNF blow-up")
        return [x | y for x in a for y in b]
    if k == "not": return [frozenset([(False, f[1][1], f[1][2])])]
    return [frozenset([(True, f[1], f[2])])]

def to_clauses(f):
    g = _skolem(_nnf(_elim(f)), {}, [])
    out = []
    for c in _cnf(g):
        if any((not s, p, a) in c for (s, p, a) in c): continue        # tautology
        out.append(c)
    return out

# =====================================================================
# 3. Unification + resolution (given-clause loop, pluggable search strategy)
# =====================================================================
def _isvar(t): return isinstance(t, str) and t.startswith("?")

def _walk(t, s):
    while _isvar(t) and t in s: t = s[t]
    return t

def _occurs(v, t, s):
    t = _walk(t, s)
    if t == v: return True
    return isinstance(t, tuple) and any(_occurs(v, a, s) for a in t[1:])

def unify(a, b, s):
    a, b = _walk(a, s), _walk(b, s)
    if a == b: return s
    if _isvar(a):
        if _occurs(a, b, s): return None
        s = dict(s); s[a] = b; return s
    if _isvar(b): return unify(b, a, s)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def _apply(t, s):
    t = _walk(t, s)
    if isinstance(t, tuple): return (t[0],) + tuple(_apply(a, s) for a in t[1:])
    return t

def _size(t): return 1 + sum(_size(a) for a in t[1:]) if isinstance(t, tuple) else 1

def _canon(clause):
    """canonical key of a clause up to variable renaming (for duplicate detection)."""
    m = {}
    def ren(t):
        if isinstance(t, tuple): return (t[0],) + tuple(ren(a) for a in t[1:])
        if _isvar(t):
            if t not in m: m[t] = f"?{len(m)}"
            return m[t]
        return t
    return tuple(sorted(((s, p, tuple(ren(a) for a in args)) for s, p, args in sorted(clause, key=repr)), key=repr))

_rn = itertools.count(1)
def _rename(clause):
    m = {}; suf = next(_rn)
    def ren(t):
        if isinstance(t, tuple): return (t[0],) + tuple(ren(a) for a in t[1:])
        if _isvar(t):
            if t not in m: m[t] = f"{t.split('#')[0]}#{suf}"
            return m[t]
        return t
    return tuple((s, p, tuple(ren(a) for a in args)) for s, p, args in clause)

def _factor(lits):
    """return the clause plus all single-step factors (merge two unifiable same-sign literals)."""
    outs = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            a, b = lits[i], lits[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = {}
                for x, y in zip(a[2], b[2]):
                    s = unify(x, y, s)
                    if s is None: break
                if s is not None:
                    outs.append(tuple(dict.fromkeys((sg, p, tuple(_apply(t, s) for t in args)) for sg, p, args in lits)))
    return outs

class Prover:
    """Resolution refutation. strategy: 'bfs' | 'dfs' | 'astar' decides which clause is expanded next."""
    def __init__(self, strategy="astar", max_given=3000, time_limit=1.0, max_lits=8, max_term=12):
        self.strategy, self.max_given, self.time_limit, self.max_lits, self.max_term = strategy, max_given, time_limit, max_lits, max_term
        self.stats = {"given": 0, "generated": 0}

    def _key(self, clause, depth, tick):
        if self.strategy == "bfs": return (depth, tick)                    # Breadth-first: shallowest first
        if self.strategy == "dfs": return (-depth, -tick)                   # Depth-first: deepest/newest first
        h = sum(1 + sum(_size(a) for a in args) for _, _, args in clause)   # A*: f = g + h
        return (depth + h, tick)

    def refute(self, kb_clauses, goal_clauses, trace=None):
        t0 = time.perf_counter(); tick = itertools.count()
        usable = defaultdict(list)       # (sign,pred) -> list of clause tuples that can be resolved against
        seen = set(); heap = []
        info = {}                         # clause -> (parents, rule)
        def add_usable(c):
            for l in c: usable[(l[0], l[1])].append(c)
        for c in kb_clauses:
            c = tuple(c); k = _canon(c)
            if k in seen: continue
            seen.add(k); add_usable(c); info[c] = ("premise", ())
        for c in goal_clauses:
            c = tuple(c); k = _canon(c)
            if k in seen: continue
            seen.add(k); info[c] = ("negated goal", ())
            heapq.heappush(heap, (self._key(c, 0, next(tick)), next(tick), 0, c))
        while heap and self.stats["given"] < self.max_given:
            if time.perf_counter() - t0 > self.time_limit: return None
            _, _, depth, given = heapq.heappop(heap)
            self.stats["given"] += 1
            add_usable(given)
            g = _rename(given)
            for gi, (gs, gp, ga) in enumerate(g):
                for other in list(usable[(not gs, gp)]):
                    o = _rename(other)
                    for oi, (os_, op, oa) in enumerate(o):
                        if os_ == gs or op != gp or len(oa) != len(ga): continue
                        s = {}
                        for x, y in zip(ga, oa):
                            s = unify(x, y, s)
                            if s is None: break
                        if s is None: continue
                        rest = [l for j, l in enumerate(g) if j != gi] + [l for j, l in enumerate(o) if j != oi]
                        res = tuple(dict.fromkeys((sg, p, tuple(_apply(t, s) for t in args)) for sg, p, args in rest))
                        self.stats["generated"] += 1
                        if not res:
                            if trace is not None: trace.append((given, other, "contradiction (empty clause)"))
                            return True
                        if len(res) > self.max_lits: continue
                        if any(_size(t) > self.max_term for _, _, a in res for t in a): continue
                        if any((not sg, p, a) in res for sg, p, a in res): continue          # tautology
                        for cand in [res] + _factor(res):
                            k = _canon(cand)
                            if k in seen: continue
                            seen.add(k); info[cand] = ("resolve", (given, other))
                            heapq.heappush(heap, (self._key(cand, depth + 1, next(tick)), next(tick), depth + 1, cand))
        return None if heap else False      # None = gave up (limit), False = saturated without proof

def _fmt_term(t):
    if isinstance(t, tuple): return f"{t[0]}({', '.join(_fmt_term(a) for a in t[1:])})"
    return t
def fmt_clause(c): return " v ".join(("" if s else "~") + p + (f"({', '.join(_fmt_term(a) for a in args)})" if args else "") for s, p, args in c) or "[]"

# =====================================================================
# 4. Ground DPLL (propositional SAT) on the Herbrand instances
# =====================================================================
DPLL_DEADLINE = [float("inf")]
def dpll(clauses, assignment=None):
    """clauses: list of sets of ints (DIMACS-style). Returns a model dict or None (unsatisfiable)."""
    if time.perf_counter() > DPLL_DEADLINE[0]: raise TimeoutError("DPLL time limit")
    assignment = dict(assignment or {})
    clauses = [set(c) for c in clauses]
    while True:                                                           # unit propagation
        unit = next((next(iter(c)) for c in clauses if len(c) == 1), None)
        if unit is None: break
        assignment[abs(unit)] = unit > 0
        new = []
        for c in clauses:
            if unit in c: continue
            if -unit in c:
                c = c - {-unit}
                if not c: return None
            new.append(c)
        clauses = new
    if not clauses: return assignment
    lits = {l for c in clauses for l in c}
    for l in list(lits):                                                  # pure literal elimination
        if -l not in lits:
            assignment[abs(l)] = l > 0
            clauses = [c for c in clauses if l not in c]
    if not clauses: return assignment
    cnt = defaultdict(int)
    for c in clauses:
        for l in c: cnt[abs(l)] += 1
    v = max(cnt, key=cnt.get)                                             # branching heuristic (most frequent var)
    for val in (v, -v):
        r = dpll(clauses + [{val}], assignment)
        if r is not None: return r
    return None

def _ground_terms(consts, funcs, depth):
    terms = set(consts)
    for _ in range(depth):
        terms |= {(f,) + args for f, n in funcs for args in itertools.product(sorted(terms, key=repr), repeat=n)}
    return sorted(terms, key=repr)

def _vars_of(t, out):
    if _isvar(t): out.add(t)
    elif isinstance(t, tuple):
        for a in t[1:]: _vars_of(a, out)

def dpll_refute(kb_clauses, goal_clauses, depth=1, max_ground=6000):
    """Ground the clause set over the Herbrand universe (depth-limited) and run DPLL.
    UNSAT  => KB entails the goal (sound).   SAT/limit => unknown (False)."""
    allc = [tuple(c) for c in kb_clauses] + [tuple(c) for c in goal_clauses]
    consts, funcs = set(), set()
    def scan(t):
        if isinstance(t, tuple):
            funcs.add((t[0], len(t) - 1)); [scan(a) for a in t[1:]]
        elif not _isvar(t): consts.add(t)
    for c in allc:
        for _, _, a in c: [scan(t) for t in a]
    if not consts: consts.add("c0")
    U = _ground_terms(consts, funcs, depth)
    ids = {}
    def lid(s, p, args):
        k = (p, args)
        if k not in ids: ids[k] = len(ids) + 1
        return ids[k] if s else -ids[k]
    out = []
    for c in allc:
        vs = set()
        for _, _, a in c: [_vars_of(t, vs) for t in a]
        vs = sorted(vs)
        if len(U) ** len(vs) > max_ground: return None
        for combo in itertools.product(U, repeat=len(vs)):
            m = dict(zip(vs, combo))
            out.append({lid(s, p, tuple(_subst_term(t, m) for t in a)) for s, p, a in c})
            if len(out) > max_ground: return None
    return dpll(out) is None


import difflib
FUZZY = True
def _collect(f, preds, consts):
    k = f[0]
    if k == "atom":
        preds.add((f[1], len(f[2])))
        def walk(t):
            if isinstance(t, tuple): preds_f.add((t[0], len(t) - 1)); [walk(a) for a in t[1:]]
            elif not t.startswith("?"): consts.add(t)
        for a in f[2]: walk(a)
    elif k == "not": _collect(f[1], preds, consts)
    elif k in ("and", "or", "imp", "iff", "xor"): _collect(f[1], preds, consts); _collect(f[2], preds, consts)
    else: _collect(f[2], preds, consts)
preds_f = set()

def _edit1(a, b):
    """True if a and b differ by exactly one insertion, deletion or substitution (a typo)."""
    if a == b or abs(len(a) - len(b)) > 1: return False
    if len(a) > len(b): a, b = b, a
    i = 0
    while i < len(a) and a[i] == b[i]: i += 1
    return a[i + 1:] == b[i + 1:] if len(a) == len(b) else a[i:] == b[i + 1:]

def _merge_map(names):
    """Merge names that differ by a single character (typos such as Studen/Student); keep the longer one."""
    names = sorted(names, key=lambda n: (-len(n), n)); m = {}
    for i, a in enumerate(names):
        if a in m: continue
        for b in names[i + 1:]:
            if b in m or len(a) < 5 or len(b) < 4: continue
            if _edit1(a, b): m[b] = a
    return m

def _rename_ast(f, pm, cm):
    k = f[0]
    def rt(t):
        if isinstance(t, tuple): return (t[0],) + tuple(rt(a) for a in t[1:])
        return t if t.startswith("?") else cm.get(t, t)
    if k == "atom": return ("atom", pm.get(f[1], f[1]), tuple(rt(a) for a in f[2]))
    if k == "not": return ("not", _rename_ast(f[1], pm, cm))
    if k in ("and", "or", "imp", "iff", "xor"): return (k, _rename_ast(f[1], pm, cm), _rename_ast(f[2], pm, cm))
    return (k, f[1], _rename_ast(f[2], pm, cm))

def _unify_symbols(asts):
    preds, consts = set(), set()
    for a in asts: _collect(a, preds, consts)
    pm = _merge_map({p for p, _ in preds} - {"eq"})
    cm = _merge_map(consts)
    if not pm and not cm: return asts
    return [_rename_ast(a, pm, cm) for a in asts]

# =====================================================================
# 5. Public API
# =====================================================================
def _split_formulas(text):
    return [l.strip() for l in re.split(r"\n+", text) if l.strip()]

def looks_like_fol(text):
    """True if the text is written in logic (Unicode or ASCII notation) rather than natural language."""
    t = str(text)
    if re.search(r"Bern\s*\(", t): return False                       # CounterBench prior, not logic
    if re.search(r"[∀∃∧∨→¬⊕↔]", t): return True
    return bool(re.search(r"\b(?:forall|exists)\b|->|=>|<->|&&|\|\||\s&\s|\s\|\s", t))

def fol_entails(premises, conclusion, method="auto", strategy="astar", time_limit=1.0, trace=None):
    """premises: str (one formula per line) or list; conclusion: str. True iff premises |= conclusion."""
    plist = _split_formulas(premises) if isinstance(premises, str) else list(premises)
    asts = []
    for p in plist:
        try: asts.append(parse_fol_tolerant(p))
        except (FOLSyntaxError, RecursionError, IndexError): continue      # skip malformed premise
    try: g_ast = parse_fol_tolerant(conclusion)
    except (FOLSyntaxError, RecursionError, IndexError): return False
    allast = asts + [g_ast]
    if FUZZY: allast = _unify_symbols(allast)
    asts, g_ast = allast[:-1], allast[-1]
    kb = []
    for a in asts:
        try: kb += to_clauses(a)
        except (FOLSyntaxError, RecursionError): continue
    try: goal = to_clauses(("not", g_ast))
    except (FOLSyntaxError, RecursionError): return False
    kb.append(frozenset([(True, "eq", ("?x", "?x"))]))
    kb.append(frozenset([(False, "eq", ("?x", "?y")), (True, "eq", ("?y", "?x"))]))
    DPLL_DEADLINE[0] = time.perf_counter() + max(3.0, 5 * time_limit)   # per-problem budget
    if method in ("dpll", "auto"):                      # fast ground SAT first
        r = dpll_refute(kb, goal, depth=1)
        if r is not None or method == "dpll":
            return bool(r)                              # (auto: None = grounding too large -> fall through to resolution)
    P = Prover(strategy=strategy, time_limit=time_limit)
    r = P.refute(kb, goal, trace)
    fol_entails.last_stats = P.stats
    return r is True


# ============================================================================================
#  STRUCTURAL CAUSAL MODELS (CounterBench)
# ============================================================================================
import re, itertools

def scm_parse(given, question):
    """Parse a CounterBench prompt into (mechanisms, priors, evidence, factual, interventions, target)."""
    g = given.strip()
    m = re.search(r"We know that\s*(.*?)\.\s*(?=[A-Z][a-z]*\s*~|\s*We observed|$)", g + " ", re.S)
    body = m.group(1) if m else g
    # priors:  Blaf~ Bern(0.2)  /  Blaf ~Bern(0.7)
    priors = {v: float(p) for v, p in re.findall(r"([A-Z][a-z]+)\s*~\s*Bern\(([\d.]+)\)", g)}
    # evidence: after "We observed"
    ev = {}
    mo = re.search(r"We observed\s*(.*)$", g, re.S)
    if mo:
        txt = mo.group(1).strip().rstrip(".")
        for part in re.split(r",|\band\b", txt):
            names = re.findall(r"\b[A-Z][a-z]+\b", part)
            if names:
                neg = bool(re.search(r"\bnot\b", part, re.I)) or bool(re.search(r"\bdoes not\b|\bnot occur", part, re.I))
                ev[names[0]] = 0 if neg else 1
    # ---- mechanisms ----
    segs = [s.strip() for s in re.split(r",", body) if s.strip()]
    clauses = []   # [left_expr, effects]; left_expr = (op, [(var,neg)...])
    for s in segs:
        s = re.sub(r"^and\s+", "", s).strip()
        mm = re.match(r"^(.*?)\s+(?:together\s+)?(causes?)\s+(.*)$", s)
        if mm:
            lhs, rhs = mm.group(1), mm.group(3)
            op = "or" if re.search(r"\bor\b", lhs) else "and"
            lits = [(re.findall(r"[A-Z][a-z]+", p)[0], bool(re.search(r"\bnot\b", p, re.I)))
                    for p in re.split(r"\s+(?:and|or)\s+", lhs.replace("together", "").strip())]
            effs = [(re.findall(r"[A-Z][a-z]+", p)[0], bool(re.search(r"\bnot\b", p, re.I)))
                    for p in re.split(r"\s+and\s+", rhs.strip())]
            clauses.append([(op, lits), effs])
        elif clauses:   # continuation of the effect list: "Trune and Vork"
            for p in re.split(r"\s+and\s+", s):
                nm = re.findall(r"[A-Z][a-z]+", p)
                if nm: clauses[-1][1].append((nm[0], bool(re.search(r"\bnot\b", p, re.I))))
    mech = {}   # effect -> list of (op, lits, neg_effect)
    for (op, lits), effs in clauses:
        for v, neg in effs:
            mech.setdefault(v, []).append((op, lits, neg))
    # ---- question ----
    q = question.strip()
    target = re.findall(r"Would\s+([A-Z][a-z]+)\s+occur", q)
    target = target[0] if target else None
    do, factual = {}, {}
    body_q = re.sub(r"^.*?(?=Would)", "", q) if q.startswith("Assume") else q
    if q.startswith("Assume"):
        for part in re.findall(r"(not\s+)?([A-Z][a-z]+)", q.split("Would")[0]):
            if part[1] in ("Assume",): continue
            do[part[1]] = 0 if part[0] else 1
    else:
        mi = re.search(r"if\s+(.*?)\s+instead of\s+([A-Z][a-z]+)", q)
        if mi:
            nm = re.findall(r"(not\s+)?([A-Z][a-z]+)", mi.group(1))
            for neg, v in nm: do[v] = 0 if neg else 1
            factual[mi.group(2)] = 1 - (0 if nm[0][0] else 1) if False else 1
        else:
            mj = re.search(r"if\s+(.*?)\?", q)
            if mj:
                for neg, v in re.findall(r"(not\s+)?([A-Z][a-z]+)", mj.group(1)):
                    do[v] = 0 if neg else 1
    return mech, priors, ev, factual, do, target

def scm_answer(given, question):
    mech, priors, ev, factual, do, target = scm_parse(given, question)
    names = set(mech) | {l[0] for ms in mech.values() for (_, ls, _) in ms for l in ls} | set(ev) | set(do) | set(factual) | {target}
    roots = sorted(v for v in names if v not in mech)
    if len(roots) > 20: raise ValueError('too many exogenous variables for enumeration')

    allvars = sorted(names - set(roots))
    def evaluate(u, interv):
        # fixed-point evaluation (handles cyclic mechanisms): iterate synchronous updates from 0
        cur = {v: 0 for v in allvars}; cur.update(u); cur.update(interv)
        for _ in range(len(allvars) + 3):
            changed = False
            for v in allvars:
                if v in interv: continue
                out = 0
                for op, lits, neg in mech[v]:
                    xs = [cur.get(a, 0) ^ int(n) for a, n in lits]
                    r = all(xs) if op == "and" else any(xs)
                    out |= (int(r) ^ int(neg))
                if cur[v] != out: cur[v] = out; changed = True
            if not changed: break
        return lambda v: cur.get(v, 0)

    def run(use_factual):
        num = den = 0.0
        for bits in itertools.product([0, 1], repeat=len(roots)):          # DFS/enumeration of exogenous worlds
            u = dict(zip(roots, bits))
            w = 1.0
            for r_, b in u.items():
                p = priors.get(r_, 0.5); w *= p if b else 1 - p
            fv = evaluate(u, {})
            if any(fv(v) != x for v, x in ev.items()): continue
            if use_factual and any(fv(v) != x for v, x in factual.items()): continue
            cv = evaluate(u, do)
            den += w; num += w * cv(target)
        return num, den
    num, den = run(True)
    if den == 0: num, den = run(False)
    return (num / den > 0.5) if den else False


# ============================================================================================
#  PREDICTOR  (routes by the FORMAT of the input, never by dataset name)
# ============================================================================================
_RULE_TEMPLATE = re.compile(
    r"^(?:given that|based on|assuming|when|if)\s+(.*?),\s*"
    r"(?:it follows that|it is evident that|we can say|we can conclude|then)\s+(.*?)\.?\s*$", re.IGNORECASE)

def normalise_context(context):
    """Drop 'Facts:'/'Rules:' headers and rewrite the 5 paraphrase templates to 'If X, then Y.'"""
    out = []
    for line in str(context).split("\n"):
        line = line.strip()
        if not line or line.lower() in ("facts:", "rules:"): continue
        m = _RULE_TEMPLATE.match(line)
        out.append(f"If {m.group(1)}, then {m.group(2)}." if m else (line if line.endswith((".", "?", "!")) else line + "."))
    return " ".join(out)

def predict(context=None, statement=None, trace=False, premises_fol=None, conclusion_fol=None, fol_method="auto"):
    """True / False.  1) logic given directly -> FOL prover   2) text with Bern(p) -> causal-model solver
       3) everything else -> parser + forward chaining.  Never raises: any failure returns False."""
    try:
        _blank = lambda x: x is None or str(x).strip().lower() in ("", "nan", "none", "null")
        if (premises_fol is not None or conclusion_fol is not None) and (_blank(premises_fol) or _blank(conclusion_fol)):
            return (False, ["empty input"]) if trace else False
        if premises_fol is None and (_blank(context) or _blank(statement)):
            return (False, ["empty input"]) if trace else False
        if premises_fol is not None and conclusion_fol is not None:
            return bool(fol_entails(str(premises_fol), str(conclusion_fol), method=fol_method))
        if context is not None and statement is not None and looks_like_fol(str(context)):
            return bool(fol_entails(str(context), str(statement), method=fol_method))
        if re.search(r"Bern\s*\(", str(context)):
            return bool(scm_answer(str(context), str(statement)))
        res = v_infer_answer(normalise_context(context), str(statement), trace=trace)
        return (bool(res[0]), res[1]) if trace else bool(res[0] if isinstance(res, tuple) else res)
    except BaseException as err:                    # includes RecursionError / TimeoutError; never stop the batch
        if isinstance(err, KeyboardInterrupt): raise
        return (False, [f"(error: {err})"]) if trace else False

# ============================================================================================
#  ROBUST FILE LOADING + AUTOMATIC SCHEMA DETECTION
#  Nothing is assumed about the hidden file: not the column names, not the number of columns,
#  not even whether there is a header row. Roles (id / context / statement / gold label /
#  FOL-premises / FOL-conclusion) are inferred from the CONTENT of the columns; names are only
#  used as a tie-breaking hint. Detection looks at <= 300 rows, so it costs a few milliseconds.
# ============================================================================================
import io
try: csv.field_size_limit(2**31 - 1)
except OverflowError: csv.field_size_limit(10**9)

_EXT_KIND = {".parquet": "parquet", ".pq": "parquet", ".jsonl": "jsonl", ".ndjson": "jsonl", ".json": "json",
             ".csv": "csv", ".tsv": "tsv", ".xlsx": "excel", ".xls": "excel"}

def _read_text(path):
    raw = open(path, "rb").read()
    if raw[:2] in (b"\xff\xfe", b"\xfe\xff"): return raw.decode("utf-16")
    for enc in ("utf-8-sig", "cp1252", "latin-1"):
        try: return raw.decode(enc)
        except UnicodeDecodeError: continue

def _sniff_kind(path):
    """csv / json / jsonl / parquet / excel, decided from the content."""
    with open(path, "rb") as f: head = f.read(4096)
    if head[:4] == b"PAR1": return "parquet"
    if head[:2] == b"PK": return "excel"
    text = head.decode("utf-8-sig", errors="ignore").lstrip()
    if text.startswith("["): return "json"
    if text.startswith("{"): return "json"          # the json loader also copes with one-object-per-line files
    return "csv"

# ---------------------------------------------------------------- cell / record normalisation
def _cell(v):
    """Any JSON / parquet / pandas value -> clean string (lists become one item per line)."""
    if v is None: return ""
    if isinstance(v, bytes): return v.decode("utf-8", "ignore")
    if hasattr(v, "tolist") and not isinstance(v, str): v = v.tolist()
    if isinstance(v, float):
        if v != v: return ""
        return str(int(v)) if v.is_integer() else str(v)
    if isinstance(v, dict): return json.dumps(v, ensure_ascii=False)
    if isinstance(v, (list, tuple)): return "\n".join(s for s in (_cell(i) for i in v) if s != "")
    return str(v)

def _flatten_record(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict) and v and not prefix: out.update(_flatten_record(v, key + "."))
        else: out[key] = _cell(v)
    return out

def _records_from_json(data):
    if isinstance(data, list): return [d if isinstance(d, dict) else {"text": d} for d in data]
    if isinstance(data, dict):
        keys = sorted(data, key=lambda k: (str(k).lower() not in ("test", "data", "examples", "rows", "instances", "questions"), 0))
        for k in keys:                                        # {"data": [ {...}, {...} ]}
            v = data[k]
            if isinstance(v, list) and v and all(isinstance(x, dict) for x in v): return v
        vals = list(data.values())
        if vals and all(isinstance(v, dict) for v in vals):   # {"id1": {...}, "id2": {...}}
            return [v if "id" in v else dict({"id": k}, **v) for k, v in data.items()]
        if vals and all(isinstance(v, list) for v in vals) and len({len(v) for v in vals}) == 1:
            return [{k: data[k][i] for k in data} for i in range(len(vals[0]))]      # columnar
        return [data]
    return []

def _load_json_any(text):
    try: return _records_from_json(json.loads(text))
    except Exception:
        recs = []
        for line in text.splitlines():
            line = line.strip().rstrip(",")
            if line and line not in ("[", "]"):
                try: recs += _records_from_json(json.loads(line))
                except Exception: continue
        return recs

# ---------------------------------------------------------------- delimited text (csv / tsv / ; / |)
def _parse_delimited(text, forced=None):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    cands = [forced] if forced else [",", "\t", ";", "|"]
    sample, best = text[:300000], None
    for d in cands:
        try:
            lens = []
            for i, row in enumerate(csv.reader(io.StringIO(sample), delimiter=d)):
                if i >= 40: break
                if row: lens.append(len(row))
        except csv.Error: continue
        if len(lens) > 1 and len(sample) < len(text): lens = lens[:-1]
        if not lens: continue
        mode = max(set(lens), key=lens.count); frac = lens.count(mode) / len(lens)
        score = (frac >= 0.9 and mode >= 2, mode if frac >= 0.9 else 0)
        if best is None or score > best[0]: best = (score, d)
    d = best[1] if best else ","
    return list(csv.reader(io.StringIO(text), delimiter=d))

def _looks_like_header(row):
    cells = [c.strip() for c in row]
    if not any(cells): return False
    for c in cells:
        if (len(c) > 40 or len(c.split()) > 4 or c.endswith((".", "?", "!")) or _is_logic(c)
                or re.search(r"\w\([^)]*\)", c)): return False
    return True

def _unique_names(names, width):
    out, seen = [], {}
    for i in range(width):
        n = names[i].strip() if i < len(names) else ""
        n = n or f"col{i}"
        if n in seen: seen[n] += 1; n = f"{n}_{seen[n]}"
        else: seen[n] = 1
        out.append(n)
    return out

def _rows_from_table(table):
    table = [r for r in table if any(c.strip() for c in r)]
    if not table: return []
    width = max(len(r) for r in table[:200])
    if _looks_like_header(table[0]): names, body = table[0], table[1:]
    else: names, body = [], table                        # header-less file: columns become col0, col1, ...
    names = _unique_names(names, width)
    return [{names[i]: (r[i] if i < len(r) else "") for i in range(width)} for r in body]

def load_rows(path):
    """Return a list of {column: string} rows from csv / tsv / json / jsonl / parquet / xlsx."""
    p = str(path); ext = os.path.splitext(p.lower())[1]
    first = _EXT_KIND.get(ext) or _sniff_kind(p)
    last_err = None
    for kind in dict.fromkeys([first, _sniff_kind(p), "csv", "json", "parquet"]):
        try:
            if kind == "parquet":
                import pandas as pd; recs = pd.read_parquet(p).to_dict("records")
            elif kind == "excel":
                import pandas as pd; recs = pd.read_excel(p, dtype=object).to_dict("records")
            elif kind in ("json", "jsonl"): recs = _load_json_any(_read_text(p))
            else: recs = _rows_from_table(_parse_delimited(_read_text(p), "\t" if kind == "tsv" else None))
            recs = [_flatten_record(r) for r in recs]
            if recs: return recs
        except Exception as e: last_err = e
    if last_err: raise last_err
    return []

# ---------------------------------------------------------------- web links / local lookup
def _direct_url(u):
    """Turn common 'share' links into direct-download links."""
    u = u.strip()
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/blob/(.*)", u)
    if m: return f"https://raw.githubusercontent.com/{m.group(1)}/{m.group(2)}/{m.group(3)}"
    if "huggingface.co" in u and "/blob/" in u: return u.replace("/blob/", "/resolve/", 1)
    m = re.search(r"drive\.google\.com/file/d/([\w-]+)", u) or re.search(r"drive\.google\.com/(?:open|uc)\?[^#]*id=([\w-]+)", u)
    if m: return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "dropbox.com" in u:
        u = re.sub(r"([?&])dl=0", r"\1dl=1", u)
        return u if "dl=1" in u else u + ("&" if "?" in u else "?") + "dl=1"
    return u

def fetch_to_local(link):
    """Local path -> returned unchanged. Web link -> downloaded to a temporary file; returns its path."""
    link = str(link).strip().strip('"').strip("'")
    if not re.match(r"https?://", link, re.I): return link
    url = _direct_url(link)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r: data = r.read()
    if data[:15].lower().lstrip().startswith((b"<!doctype html", b"<html")):
        raise ValueError("the link returned a web page, not a data file (use a direct download / 'raw' link)")
    ext = os.path.splitext(urllib.parse.urlparse(url).path)[1].lower()
    ext = ext if ext in _EXT_KIND else ""
    path = os.path.join(tempfile.gettempdir(), "hidden_test_download" + ext)
    with open(path, "wb") as f: f.write(data)
    print(f"Downloaded {len(data)/1024:.1f} KB from {url}")
    return path

def find_hidden_file():
    """Resolve HIDDEN_TEST_LINK (download it if it is a web link). Falls back to common local names."""
    link = str(HIDDEN_TEST_LINK).strip()
    if link:
        try:
            p = fetch_to_local(link)
            if os.path.isfile(p): return p
            print(f"[!] file not found: {link}")
        except Exception as e:
            print(f"[!] could not fetch {link}: {type(e).__name__}: {e}")
    for folder in (".", "datasets/tests", "tests", "data"):
        for stem in ("private_test", "hidden_test", "test", "test_data", "evaluation", "eval"):
            for ext in (".csv", ".json", ".jsonl", ".parquet", ".tsv", ".xlsx"):
                p = os.path.join(folder, stem + ext)
                if os.path.isfile(p): print(f"Using {p} found locally"); return p
    return None

def _to_bool(x):
    return str(x).strip().lower() in ("true", "yes", "1", "entailment", "entailed", "t", "y", "correct", "valid")

# ============================================================================================
#  AUTOMATIC SCHEMA DETECTION
# ============================================================================================
_LABEL_VOCAB = {"true", "false", "yes", "no", "1", "0", "t", "f", "y", "n", "entailment", "not entailment", "not_entailment",
                "entailed", "not entailed", "contradiction", "neutral", "uncertain", "unknown", "correct", "incorrect",
                "valid", "invalid", "proved", "disproved", "entails", "not entails"}
_CTX_NAMES  = {"context", "premises", "premise", "given", "given_info", "info", "kb", "knowledge", "knowledge_base", "facts",
               "theory", "passage", "background", "story", "scenario", "text", "rules", "problem", "paragraph", "description", "input"}
_QRY_NAMES  = {"statement", "question", "query", "conclusion", "hypothesis", "claim", "goal", "proposition", "queries", "q"}
_GOLD_NAMES = {"label", "labels", "flag", "answer", "gold", "gold_label", "target", "truth", "output", "y", "expected",
               "result", "class", "verdict", "ground_truth", "entailment"}
_ID_LAST    = {"id", "ids", "idx", "index", "uid", "qid", "uuid", "key", "no", "num", "number", "sno", "serial"}
_META_LAST  = {"config", "split", "source", "dataset", "depth", "category", "type", "difficulty", "domain", "topic", "proof",
               "explanation", "rationale", "trace", "hops", "reasoning", "subset", "group", "version", "language", "lang",
               "length", "count", "style", "template", "fold", "note", "notes", "comment", "metadata", "tags", "author",
               "url", "link", "timestamp", "date", "level", "size", "score", "weight", "theory_id", "story_id", "name"}
_ATOM_LINE = re.compile(r"^[¬~!\-\s]*\(*\s*[A-Za-z_]\w*\s*\(.*\)\s*\)*\.?$")
_LOGIC_SYM = re.compile(r"[∀∃∧∨→¬⊕↔⊃≡]|<->|->|=>|&&|\|\||\s&\s|\b(?:forall|exists)\s+[A-Za-z_]\w*\s*[.(,:]", re.I)

def _tokens(name):
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(name))
    return [t for t in re.split(r"[^a-z0-9]+", s.lower()) if t]

def _hint(name, vocab):
    """3 = whole name is a known word, 2 = one of its words is, 0 = no hint."""
    t = _tokens(name)
    if not t: return 0
    if "_".join(t) in vocab: return 3
    return 2 if any(w in vocab for w in t) else 0

def _is_logic(v):
    v = str(v)
    if not v.strip() or re.search(r"Bern\s*\(", v): return False
    if _LOGIC_SYM.search(v): return True
    lines = [l.strip() for l in v.split("\n") if l.strip()]
    return bool(lines) and all(_ATOM_LINE.match(l) for l in lines)

def _profile(col, vals):
    nz = [str(v).strip() for v in vals if v is not None and str(v).strip() != ""]
    n = len(nz)
    P = {"col": col, "n": n, "fill": n / max(len(vals), 1)}
    if n == 0: return dict(P, avg_len=0, avg_words=0, uniq=0, distinct=0, frac_label=0, frac_logic=0, frac_bern=0, frac_q=0, frac_num=0, frac_multi=0)
    low = [v.lower() for v in nz]
    P.update(avg_len=sum(map(len, nz)) / n, avg_words=sum(len(v.split()) for v in nz) / n,
             uniq=len(set(nz)) / n, distinct=len(set(low)),
             frac_label=sum(v in _LABEL_VOCAB for v in low) / n,
             frac_logic=sum(_is_logic(v) for v in nz) / n,
             frac_bern=sum(bool(re.search(r"Bern\s*\(", v)) for v in nz) / n,
             frac_q=sum(v.endswith("?") for v in nz) / n,
             frac_num=sum(bool(re.fullmatch(r"-?\d+(?:\.\d+)?", v)) for v in nz) / n,
             frac_multi=sum("\n" in v for v in nz) / n)
    return P

def _resolve_col(keys, spec):
    if spec is None: return None
    if isinstance(spec, int) and not isinstance(spec, bool): return keys[spec] if -len(keys) <= spec < len(keys) else None
    for k in keys:
        if str(k) == str(spec): return k
    for k in keys:
        if "_".join(_tokens(k)) == "_".join(_tokens(spec)): return k
    return None

_STOP = {"the", "a", "an", "is", "are", "was", "were", "it", "if", "then", "of", "and", "or", "not", "that", "to", "be", "in", "on", "x", "all", "some", "no", "does", "do", "has", "have"}
def _vocab(s): return {w for w in re.findall(r"[a-z0-9]+", str(s).lower()) if w not in _STOP}

def _pick_pair(cols, prof, fix_ctx=None, fix_qry=None, texts=None):
    """Choose (context_col, statement_col) among text columns using names (hint) + content (decisive)."""
    cols = list(cols)
    if not cols: return fix_ctx, fix_qry
    if len(cols) == 1 and not (fix_ctx or fix_qry): return cols[0], None
    lens = sorted(prof[c]["avg_len"] for c in cols)
    def rank(c): return lens.index(prof[c]["avg_len"]) / max(len(cols) - 1, 1)       # 0 shortest .. 1 longest
    def not_prop(c): return (not prof[c]["frac_logic"] >= .5) and prof[c]["avg_words"] < 2
    def c_score(c): return 3 * _hint(c, _CTX_NAMES) + 4 * rank(c) + 2 * prof[c]["frac_bern"] + 1.5 * prof[c]["frac_multi"] - (3 if not_prop(c) else 0)
    def q_score(c): return 3 * _hint(c, _QRY_NAMES) + 4 * (1 - rank(c)) + 2 * prof[c]["frac_q"] - (3 if not_prop(c) else 0)
    V = {c: [_vocab(v) for v in texts[c]] for c in cols} if texts else {}
    def overlap(c, q):
        """How much of the statement's vocabulary is found in the context, averaged over rows (0..1). A real
        context/statement pair shares entity and predicate words; an unrelated column does not."""
        if c not in V or q not in V: return 0.0
        vals = [len(a & b) / len(b) for a, b in zip(V[c], V[q]) if b]
        return sum(vals) / len(vals) if vals else 0.0
    best, best_s = (None, None), -1e9
    for c in ([fix_ctx] if fix_ctx else cols + [None]):
        for q in ([fix_qry] if fix_qry else cols + [None]):
            if c is not None and c == q: continue
            if c is None and q is None: continue
            s = (c_score(c) if c in prof else 0) + (q_score(q) if q in prof else 0)
            if c is None or q is None: s -= 6                                      # prefer a real pair
            else: s += 6 * overlap(c, q)
            if s > best_s: best, best_s = (c, q), s
    return best

def detect_schema(rows, overrides=None, verbose=True):
    """Infer the role of every column. Returns {'ctx','qry','pf','cf','id','gold','report'} (values are column names or None)."""
    overrides = overrides or {}
    keys = []
    for r in rows[:300]:
        for k in r:
            if k not in keys: keys.append(k)
    step = max(1, len(rows) // 300)
    sample = rows[::step][:300]
    prof = {k: _profile(k, [r.get(k, "") for r in sample]) for k in keys}
    texts = {k: [str(r.get(k, "")) for r in sample[:100]] for k in keys}        # vocabulary check uses <= 100 rows
    role, why = {}, {}
    forced = {name: _resolve_col(keys, overrides.get(name)) for name in ("context", "statement", "premises_fol", "conclusion_fol", "id", "label")}
    for name, col in forced.items():
        if col is not None: role[col] = name; why[col] = "set by hand (COLUMN_OVERRIDES)"
    free = [k for k in keys if k not in role]

    # 1) empty columns
    for k in list(free):
        if prof[k]["n"] == 0: role[k] = "ignored"; why[k] = "empty"; free.remove(k)

    # 2) gold label: a column whose values are (almost) all true/false/yes/no/entailment-style words
    if forced["label"] is None:
        cands = [(2 + _hint(k, _GOLD_NAMES) + prof[k]["frac_label"], k) for k in free
                 if prof[k]["frac_label"] >= .9 and prof[k]["distinct"] <= 6 and prof[k]["frac_logic"] < .5]
        if cands:
            k = max(cands, key=lambda t: t[0])[1]; role[k] = "label"; why[k] = "values are true/false-style labels"; free.remove(k)
    for k in list(free):                                   # other label-like or answer-named columns are never inputs
        p = prof[k]
        if (p["frac_label"] >= .9 and p["distinct"] <= 6 and p["frac_logic"] < .5) or (_hint(k, _GOLD_NAMES) == 3 and p["avg_words"] < 3):
            role[k] = "ignored"; why[k] = "label-like column"; free.remove(k)

    # 3) id column
    if forced["id"] is None:
        cands = []
        for k in free:
            p = prof[k]; t = _tokens(k)
            by_name = bool(t) and (t[-1] in _ID_LAST or t == ["unnamed", "0"] or str(k).lower().startswith("unnamed"))
            by_content = (p["n"] >= 2 and p["uniq"] >= .99 and p["avg_len"] <= 40 and p["avg_words"] <= 1.5 and p["frac_logic"] < .5)
            if (by_name and p["avg_words"] <= 3 and p["frac_logic"] < .5) or by_content:
                cands.append((3 * by_name + (t == ["id"]) + p["uniq"], k))
        if cands:
            k = max(cands, key=lambda t: t[0])[1]; role[k] = "id"; why[k] = "unique short identifier"; free.remove(k)

    # 4) metadata / numeric / code-like columns are not inputs
    nS = len(sample)
    for k in list(free):
        p = prof[k]; t = _tokens(k)
        has_hint = _hint(k, _CTX_NAMES | _QRY_NAMES)
        reason = None
        if t and t[-1] in _META_LAST: reason = "metadata (by name)"
        elif p["frac_num"] >= .9: reason = "numeric column"
        elif nS >= 20 and p["distinct"] <= max(2, 0.03 * nS) and p["avg_len"] <= 25 and p["avg_words"] < 3 and p["frac_logic"] < .5 and not has_hint: reason = "categorical metadata"
        elif p["frac_logic"] < .5 and p["avg_words"] < 1.8 and p["avg_len"] <= 25 and not has_hint: reason = "short code-like column"
        if reason: role[k] = "ignored"; why[k] = reason; free.remove(k)
    if not free:                                           # over-pruned: fall back to every non-id / non-label column
        free = [k for k in keys if k not in role or role[k] == "ignored" and prof[k]["n"] > 0 and "label" not in why.get(k, "")]
        for k in free: role.pop(k, None)

    # 5) text pool -> FOL pair and / or natural-language pair
    fol = [k for k in free if prof[k]["frac_logic"] >= .5]
    nl = [k for k in free if k not in fol]
    ctx = qry = pf = cf = None
    if forced["premises_fol"] or forced["conclusion_fol"] or len(fol) >= 2:
        pf, cf = _pick_pair([k for k in fol if k not in (forced["premises_fol"], forced["conclusion_fol"])] or fol, prof, forced["premises_fol"], forced["conclusion_fol"], texts)
        if len(nl) >= 2 or forced["context"] or forced["statement"]:
            ctx, qry = _pick_pair(nl, prof, forced["context"], forced["statement"], texts)
    else:
        ctx, qry = _pick_pair(free, prof, forced["context"], forced["statement"], texts)
    if forced["context"] and not ctx: ctx = forced["context"]
    if forced["statement"] and not qry: qry = forced["statement"]
    for k, nm in ((ctx, "CONTEXT"), (qry, "STATEMENT"), (pf, "PREMISES (logic)"), (cf, "CONCLUSION (logic)")):
        if k is not None and k not in role: role[k] = nm; why[k] = "chosen from content"
    for k, nm in ((ctx, "CONTEXT"), (qry, "STATEMENT"), (pf, "PREMISES (logic)"), (cf, "CONCLUSION (logic)")):
        if k is not None and role.get(k) in ("context", "statement", "premises_fol", "conclusion_fol"): role[k] = nm
    gold = next((k for k, v in role.items() if v in ("label",)), None)
    idc = next((k for k, v in role.items() if v in ("id",)), None)
    S = {"ctx": ctx, "qry": qry, "pf": pf, "cf": cf, "id": idc, "gold": gold}
    if verbose:
        print(f"Schema detected automatically from {len(rows)} rows x {len(keys)} columns:")
        for k in keys:
            ex = (str(sample[0].get(k, "")).replace("\n", " | ")[:48]) if sample else ""
            print(f"   {str(k)[:26]:26s} -> {role.get(k, 'unused'):18s} e.g. {ex!r}" + (f"   [{why[k]}]" if k in why and role.get(k) == 'ignored' else ""))
    return S

def _split_single(text):
    """Only one text column was found: try to split 'context ... Conclusion: statement' (else last line = statement)."""
    text = str(text)
    m = re.search(r"(?im)^\s*(?:conclusion|question|statement|hypothesis|query|claim)\s*[:\-]\s*(.+)$", text)
    if m: return text[:m.start()].strip(), m.group(1).strip()
    lines = [l for l in text.split("\n") if l.strip()]
    return ("\n".join(lines[:-1]), lines[-1]) if len(lines) >= 2 else (None, None)

def _predict_row(r, S):
    """One row -> bool. Uses the logic columns if filled, otherwise the natural-language ones."""
    try:
        if S["pf"] and S["cf"]:
            p, c = r.get(S["pf"], ""), r.get(S["cf"], "")
            if str(p).strip() and str(c).strip(): return predict(premises_fol=p, conclusion_fol=c)
        if S["ctx"] is None and S["qry"] is None and S["pf"] and not S["cf"]: return predict(context=r.get(S["pf"], ""), statement=_split_single(r.get(S["pf"], ""))[1])
        ctx = r.get(S["ctx"], "") if S["ctx"] else ""
        q = r.get(S["qry"], "") if S["qry"] else ""
        if not str(q).strip():
            c2, q2 = _split_single(ctx)
            if c2 is None: return False
            ctx, q = c2, q2
        return predict(context=ctx, statement=q)
    except Exception:
        return False

# ============================================================================================
#  MAIN: private test file -> predictions -> submission.csv
# ============================================================================================
def main():
    path = find_hidden_file(); rows = []
    if path:
        try: rows = load_rows(path); print(f"Loaded test file: {path}  ({len(rows)} rows)")
        except Exception as e: print(f"[!] could not read {path}: {type(e).__name__}: {e}")
    if not rows: print("[!] no test rows available - set HIDDEN_TEST_LINK in section 0. An empty submission file is written.")
    t_det = time.perf_counter()
    try: S = detect_schema(rows, COLUMN_OVERRIDES) if rows else {"ctx": None, "qry": None, "pf": None, "cf": None, "id": None, "gold": None}
    except Exception:
        print("[!] schema detection failed, using plain name matching:"); traceback.print_exc()
        keys = list(rows[0].keys()); S = {"ctx": _pick(keys, ["context", "premises"]), "qry": _pick(keys, ["statement", "question", "conclusion"]), "pf": None, "cf": None, "id": _pick(keys, ["id"]), "gold": _pick(keys, ["label"])}
    det_time = time.perf_counter() - t_det
    idc, gold_c = S["id"], S["gold"]

    t0 = time.perf_counter()
    labels = [_predict_row(r, S) for r in rows]
    infer_time = time.perf_counter() - t0

    out_rows = [(r.get(idc, i) if idc else i, "True" if l else "False") for i, (r, l) in enumerate(zip(rows, labels))]
    try:
        os.makedirs(os.path.dirname(SUBMISSION_FILE) or ".", exist_ok=True)
        with open(SUBMISSION_FILE, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f); w.writerow(["id", "label"]); w.writerows(out_rows)
        print(f"Wrote {SUBMISSION_FILE}  ({len(out_rows)} rows)")
    except Exception as e:
        print("[!] could not write", SUBMISSION_FILE, "->", e)
        try:
            with open("submission.csv", "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f); w.writerow(["id", "label"]); w.writerows(out_rows)
            print("Wrote ./submission.csv instead")
        except Exception as e2: print("[!] fallback write failed too:", e2)
    n = max(len(rows), 1)
    print(f"Schema detection time: {1000*det_time:.1f} ms")
    print(f"Total inference time: {infer_time:.2f}s  ({1000*infer_time/n:.2f} ms / query)")
    if gold_c and rows:
        try:
            acc = sum(l == _to_bool(r.get(gold_c)) for r, l in zip(rows, labels)) / len(rows)
            print(f"Accuracy on this file: {100*acc:.2f}%")
        except Exception: pass

try:
    main()
except Exception:
    print("Unexpected error (the cell did not crash):"); traceback.print_exc()

[!] file not found: private_test.csv
[!] no test rows available - set HIDDEN_TEST_LINK in section 0. An empty submission file is written.
Wrote submission/submission.csv  (0 rows)
Schema detection time: 0.0 ms
Total inference time: 0.00s  (0.00 ms / query)
